In [1]:
#!/usr/bin/env python3

import io
import sqlite3
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from scipy.ndimage import gaussian_filter
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

NSIDE_EXPECTED = 32
NEST = True
UNSEEN = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan
RUBIN_LAT = -30.244639
RUBIN_LON = -70.749417
RUBIN_HEIGHT_M = 2663.0
CBAR_VMIN = -0.2
CBAR_VMAX = 2.0
BIN_SIZE_KM = 1000
R_PROJECTION = 10000.0

SNAPSHOT_TIMES = [
    "2025-07-02 02:17:07",
    "2025-07-02 02:21:49",
    "2025-07-02 03:21:24",
    "2025-07-02 03:30:56",
    "2025-07-02 03:43:50",
    "2025-07-02 04:23:10",
    "2025-07-02 04:30:32",
    "2025-07-02 07:11:10",
]

def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url

def fetch_map_resourcepath(url: str):
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    bio = io.BytesIO(data)
    with h5py.File(bio, "r") as f:
        if "clouds" not in f:
            raise RuntimeError(f"'clouds' not found in {url}")
        arr = np.array(f["clouds"]).ravel()
    return arr

def healpix_radec_to_ortho(mp, nside, nest=NEST,
                           observer_lat=RUBIN_LAT, observer_lon=RUBIN_LON,
                           observer_height_m=RUBIN_HEIGHT_M, obstime=None):
    if nside is None:
        nside = hp.get_nside(mp)
    npix = hp.nside2npix(nside)
    pix = np.arange(npix)
    theta_pix, phi_pix = hp.pix2ang(nside, pix, nest=nest)
    ra = np.degrees(phi_pix)
    dec = 90.0 - np.degrees(theta_pix)
    if obstime is None:
        obstime = Time.now().utc
    else:
        obstime = Time(obstime) if not isinstance(obstime, Time) else obstime
        obstime = obstime.utc
    location = EarthLocation(lat=observer_lat * u.deg,
                             lon=observer_lon * u.deg,
                             height=observer_height_m * u.m)
    sky = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
    altaz = sky.transform_to(AltAz(obstime=obstime, location=location))
    alt_deg = altaz.alt.deg
    az_deg = altaz.az.deg % 360.0
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)
    return az_deg, alt_deg, vals

def radec_to_altaz_xy(ra_deg, dec_deg, obstime,
                      observer_lat=RUBIN_LAT, observer_lon=RUBIN_LON,
                      observer_height_m=RUBIN_HEIGHT_M):
    if not isinstance(obstime, Time):
        obstime = Time(obstime)
    obstime = obstime.utc

    location = EarthLocation(lat=observer_lat * u.deg,
                             lon=observer_lon * u.deg,
                             height=observer_height_m * u.m)
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    altaz = sky.transform_to(AltAz(obstime=obstime, location=location))

    alt_deg = altaz.alt.deg
    az_deg = altaz.az.deg % 360.0

    alt_rad = np.radians(alt_deg)
    az_rad = np.radians(az_deg)

    x_altaz = -np.cos(alt_rad) * np.sin(az_rad)
    y_altaz = np.cos(alt_rad) * np.cos(az_rad)
    z_altaz = np.sin(alt_rad)

    if z_altaz > 0.01:
        scale = R_PROJECTION / z_altaz
        x_km = x_altaz * scale
        y_km = y_altaz * scale
        return x_km, y_km, alt_deg, az_deg
    else:
        return None, None, alt_deg, az_deg

def process_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    az, alt, vals = healpix_radec_to_ortho(mp, NSIDE_EXPECTED, obstime=obstime)

    alt_rad = np.radians(alt)
    az_rad = np.radians(az)

    x_altaz = -np.cos(alt_rad) * np.sin(az_rad)
    y_altaz = np.cos(alt_rad) * np.cos(az_rad)
    z_altaz = np.sin(alt_rad)

    mask = z_altaz > 0.01
    scale = np.where(mask, R_PROJECTION / z_altaz, np.nan)

    x_flat = x_altaz * scale
    y_flat = y_altaz * scale
    vals_flat = np.where(mask, vals, np.nan)

    radius = np.sqrt(x_flat**2 + y_flat**2)
    crop_mask = radius <= 15000.0

    x_cropped = x_flat[crop_mask]
    y_cropped = y_flat[crop_mask]
    vals_cropped = vals_flat[crop_mask]

    valid_mask = ~np.isnan(vals_cropped)
    x_valid = x_cropped[valid_mask]
    y_valid = y_cropped[valid_mask]
    vals_valid = vals_cropped[valid_mask]

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)

    H_sum, _, _ = np.histogram2d(x_valid, y_valid, bins=[x_edges, y_edges], weights=vals_valid)
    H_count, _, _ = np.histogram2d(x_valid, y_valid, bins=[x_edges, y_edges])

    with np.errstate(divide='ignore', invalid='ignore'):
        H_avg = H_sum / H_count
        H_avg[H_count == 0] = np.nan

    H_avg = H_avg.T

    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    X_grid, Y_grid = np.meshgrid(x_centers, y_centers)

    R_grid = np.sqrt(X_grid**2 + Y_grid**2)
    H_avg[R_grid > 15000] = np.nan

    return H_avg, X_grid, Y_grid, x_edges, y_edges

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)

    g1_smooth = gaussian_filter(g1, sigma=sigma)
    g2_smooth = gaussian_filter(g2, sigma=sigma)

    mask1 = ~np.isnan(grid1) & (grid1 != 0)
    mask2 = ~np.isnan(grid2) & (grid2 != 0)

    best_corr = -np.inf
    best_dx, best_dy = 0, 0

    for dy in range(-search_range, search_range + 1):
        for dx in range(-search_range, search_range + 1):
            shifted = ndimage.shift(g2_smooth, (dy, dx), order=1, mode='constant', cval=0)
            shifted_mask = ndimage.shift(mask2.astype(float), (dy, dx), order=0, mode='constant', cval=0) > 0.5

            valid = mask1 & shifted_mask

            if np.sum(valid) < 100:
                continue

            v1 = g1_smooth[valid]
            v2 = shifted[valid]

            if len(v1) > 0 and np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0, 1]
                if corr > best_corr:
                    best_corr = corr
                    best_dx = dx
                    best_dy = dy

    return best_dx, best_dy, best_corr

def find_absolute_minimum(grid):
    valid_mask = ~np.isnan(grid)
    if not np.any(valid_mask):
        return 0, 0, np.nan

    min_idx = np.nanargmin(grid)
    y_idx, x_idx = np.unravel_index(min_idx, grid.shape)

    x_km = (x_idx - 15) * BIN_SIZE_KM
    y_km = (y_idx - 15) * BIN_SIZE_KM
    min_val = grid[y_idx, x_idx]

    return x_km, y_km, min_val

def predict_future_position(current_x, current_y, dx_pix, dy_pix):
    dx_km = dx_pix * BIN_SIZE_KM
    dy_km = dy_pix * BIN_SIZE_KM

    new_x = current_x - dx_km
    new_y = current_y - dy_km

    max_radius = 14000
    radius = np.sqrt(new_x**2 + new_y**2)
    if radius > max_radius:
        scale = max_radius / radius
        new_x *= scale
        new_y *= scale

    return new_x, new_y

def get_value_at_position(grid, x_km, y_km):
    x_idx = int((x_km / BIN_SIZE_KM) + 15)
    y_idx = int((y_km / BIN_SIZE_KM) + 15)

    if 0 <= x_idx < grid.shape[1] and 0 <= y_idx < grid.shape[0]:
        return grid[y_idx, x_idx]
    else:
        return np.nan

def load_night_data(csv_file, target_date="2025-07-02"):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', '').str.strip()

    time_col = 'time'
    url_col = 'lsst.sal.DREAM.logevent_largeFileObjectAvailable.url'

    df = df.dropna(subset=[url_col]).copy()
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce', utc=True)
    df = df.dropna(subset=[time_col]).copy()
    df = df[df[url_col].str.contains('.hdf5', case=False, na=False)].copy()
    df = df[df[url_col].str.contains('cloud_sys', case=False, na=False)].copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    start = pd.Timestamp(f"{target_date} 00:00:00", tz="UTC")
    end = pd.Timestamp(f"{target_date} 23:59:59", tz="UTC")
    df_night = df[(df[time_col] >= start) & (df[time_col] <= end)].copy()

    print(f"Loaded {len(df_night)} candidate frames for {target_date}")
    return df_night

def load_scheduler_observations(db_file):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]

    table_name = None
    for possible_name in ['observations', 'SummaryAllProps', 'Summary', 'obs']:
        if possible_name in tables:
            table_name = possible_name
            break
    if table_name is None:
        table_name = tables[0]

    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()

    night_col = None
    for col in obs_df.columns:
        if 'night' in col.lower():
            night_col = col
            break
    if night_col is None:
        raise RuntimeError("Could not find night column")

    obs_df['night'] = obs_df[night_col].astype(int)

    for req in ['observationStartMJD', 'fieldRA', 'fieldDec']:
        found = False
        for col in obs_df.columns:
            if req.lower() in col.lower():
                if col != req:
                    obs_df[req] = obs_df[col]
                found = True
                break
        if not found:
            raise RuntimeError(f"Could not find required column: {req}")

    best_night = obs_df.groupby('night').size().idxmax()
    night_obs = obs_df[obs_df['night'] == best_night].copy()
    night_obs = night_obs.sort_values('observationStartMJD').reset_index(drop=True)

    print(f"Using scheduler night {best_night} with {len(night_obs)} observations")
    return night_obs

def match_scheduler_to_frames(scheduler_obs, frame_times):
    n_frames = len(frame_times)
    frame_fractions = np.linspace(0, 1, n_frames)

    mjd_vals = scheduler_obs['observationStartMJD'].values
    mjd_min = mjd_vals.min()
    mjd_max = mjd_vals.max()
    mjd_range = mjd_max - mjd_min

    if mjd_range > 0:
        scheduler_fractions = (mjd_vals - mjd_min) / mjd_range
    else:
        scheduler_fractions = np.zeros(len(mjd_vals))

    matched_obs = []
    for frame_frac in frame_fractions:
        diffs = np.abs(scheduler_fractions - frame_frac)
        closest_idx = np.argmin(diffs)
        matched_obs.append(scheduler_obs.iloc[closest_idx])

    return matched_obs

def find_nearest_frames(df, requested_times):
    requested = pd.to_datetime(requested_times, utc=True)
    selected_rows = []

    for t in requested:
        idx = np.argmin(np.abs(df['time'] - t))
        row = df.iloc[idx].copy()
        row['requested_time'] = t
        row['time_offset_sec'] = abs((row['time'] - t).total_seconds())
        selected_rows.append(row)

    out = pd.DataFrame(selected_rows).drop_duplicates(subset=['time']).reset_index(drop=True)
    print("\nSelected frames:")
    for _, row in out.iterrows():
        print(f"  requested {row['requested_time']}  ->  matched {row['time']}  (Δ={row['time_offset_sec']:.1f} s)")
    return out

def build_all_snapshot_data(df_selected, scheduler_obs):
    all_grids = []
    all_metas = []
    X_grid = Y_grid = x_edges = y_edges = None

    for _, row in df_selected.iterrows():
        url = transform_url(row['lsst.sal.DREAM.logevent_largeFileObjectAvailable.url'])
        mp = fetch_map_resourcepath(url)
        obstime = row['time'].to_pydatetime()
        grid, Xg, Yg, xe, ye = process_to_grid(mp, obstime)

        all_grids.append(grid)
        all_metas.append({
            "url": url,
            "time": obstime,
            "requested_time": row['requested_time']
        })

        if X_grid is None:
            X_grid, Y_grid, x_edges, y_edges = Xg, Yg, xe, ye

    matched_scheduler = match_scheduler_to_frames(scheduler_obs, [m['time'] for m in all_metas])

    x_abs, y_abs, _ = find_absolute_minimum(all_grids[0])
    x_pred, y_pred = x_abs, y_abs

    abs_positions = [(x_abs, y_abs)]
    pred_positions = [(x_pred, y_pred)]
    motion_vectors = [(0, 0)]

    history_length = 3

    for i in range(1, len(all_grids)):
        x_abs, y_abs, _ = find_absolute_minimum(all_grids[i])
        abs_positions.append((x_abs, y_abs))

        if i >= history_length:
            dx_total, dy_total = 0, 0
            count = 0
            for j in range(1, history_length + 1):
                dx, dy, conf = compute_motion_with_correlation(
                    all_grids[i-j], all_grids[i-j+1], sigma=5.0, search_range=10
                )
                if conf > 0.5:
                    dx_total += dx
                    dy_total += dy
                    count += 1
            if count > 0:
                dx_avg = dx_total / count
                dy_avg = dy_total / count
            else:
                dx_avg, dy_avg = 0, 0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(
                all_grids[i-1], all_grids[i], sigma=5.0, search_range=10
            )

        motion_vectors.append((dx_avg, dy_avg))
        x_pred, y_pred = predict_future_position(x_pred, y_pred, dx_avg, dy_avg)
        pred_positions.append((x_pred, y_pred))

    return {
        "all_grids": all_grids,
        "all_metas": all_metas,
        "matched_scheduler": matched_scheduler,
        "abs_positions": abs_positions,
        "pred_positions": pred_positions,
        "motion_vectors": motion_vectors,
        "x_edges": x_edges,
        "y_edges": y_edges,
    }

def plot_snapshot_grid(snapshot_data, output_file="cloud_tracking_snapshots_jul02.png"):
    all_grids = snapshot_data["all_grids"]
    all_metas = snapshot_data["all_metas"]
    matched_scheduler = snapshot_data["matched_scheduler"]
    abs_positions = snapshot_data["abs_positions"]
    pred_positions = snapshot_data["pred_positions"]
    motion_vectors = snapshot_data["motion_vectors"]
    x_edges = snapshot_data["x_edges"]
    y_edges = snapshot_data["y_edges"]

    fig, axes = plt.subplots(2, 4, figsize=(24, 12), constrained_layout=True)
    axes = axes.flatten()

    im = None

    for i, ax in enumerate(axes):
        if i >= len(all_grids):
            ax.axis('off')
            continue

        grid = all_grids[i]
        im = ax.pcolormesh(
            x_edges, y_edges, grid,
            cmap="viridis", vmin=CBAR_VMIN, vmax=CBAR_VMAX,
            shading='flat'
        )

        theta = np.linspace(0, 2*np.pi, 200)
        ax.plot(15000 * np.cos(theta), 15000 * np.sin(theta), 'k-', linewidth=1.5, alpha=0.5)

        x_abs, y_abs = abs_positions[i]
        val_abs = get_value_at_position(grid, x_abs, y_abs)
        ax.plot(x_abs, y_abs, 'go', markersize=12, markeredgewidth=2, markeredgecolor='white', zorder=10)

        x_pred, y_pred = pred_positions[i]
        val_pred = get_value_at_position(grid, x_pred, y_pred)
        ax.plot(x_pred, y_pred, 'bo', markersize=12, markeredgewidth=2, markeredgecolor='white', zorder=10)

        obs = matched_scheduler[i]
        x_sched, y_sched, alt, az = radec_to_altaz_xy(obs['fieldRA'], obs['fieldDec'], all_metas[i]['time'])
        val_sched = np.nan

        if x_sched is not None and y_sched is not None:
            val_sched = get_value_at_position(grid, x_sched, y_sched)
            ax.plot(x_sched, y_sched, 'ro', markersize=12, markeredgewidth=2, markeredgecolor='white', zorder=10)

        if i > 0:
            dx_pix, dy_pix = motion_vectors[i]
            dx_km = dx_pix * BIN_SIZE_KM
            dy_km = dy_pix * BIN_SIZE_KM
            mag = np.sqrt(dx_km**2 + dy_km**2)

            if mag > 300:
                arrow = FancyArrowPatch(
                    (x_pred, y_pred),
                    (x_pred + dx_km, y_pred + dy_km),
                    arrowstyle='->',
                    mutation_scale=18,
                    linewidth=2,
                    color='cyan',
                    alpha=0.9,
                    zorder=15
                )
                ax.add_patch(arrow)

        ax.plot(0, 0, 'r+', markersize=10, markeredgewidth=2)

        label = (
            f"Abs: {val_abs:.2f}\n"
            f"Track: {val_pred:.2f}\n"
            f"Sched: {val_sched:.2f}" if not np.isnan(val_sched)
            else f"Abs: {val_abs:.2f}\nTrack: {val_pred:.2f}\nSched: off-grid"
        )

        ax.text(
            0.02, 0.98, label,
            transform=ax.transAxes,
            fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85)
        )

        time_str = pd.Timestamp(all_metas[i]["time"]).strftime("%H:%M:%S")
        ax.set_title(time_str, fontsize=13, weight='bold')

        ax.set_xlim(-16000, 16000)
        ax.set_ylim(-16000, 16000)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)

        if i % 4 == 0:
            ax.set_ylabel("Y (km)")
        if i >= 4:
            ax.set_xlabel("X (km)")

    cbar = fig.colorbar(im, ax=axes, shrink=0.92, pad=0.02)
    cbar.set_label("Cloud extinction (mag)", fontsize=12)

    handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='g', markeredgecolor='white', markersize=10, label='Absolute minimum'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='b', markeredgecolor='white', markersize=10, label='Motion tracking'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='r', markeredgecolor='white', markersize=10, label='Scheduler'),
        plt.Line2D([0], [0], marker='+', color='r', markersize=10, linestyle='None', label='Zenith')
    ]
    fig.legend(handles=handles, loc='upper center', ncol=4, fontsize=11, frameon=True)

    fig.suptitle("Cloud Motion Tracking vs Scheduler — Selected DREAM Snapshots (2025-07-02)", fontsize=18, weight='bold')
    plt.savefig(output_file, dpi=250, bbox_inches='tight')
    plt.close()
    print(f"\nSaved figure: {output_file}")

def main():
    csv_file = "feb5_data.csv"
    db_file = "baseline_v5.1.0_10yrs.db"

    df = load_night_data(csv_file, target_date="2025-07-02")
    if len(df) == 0:
        print("No matching cloud frames found.")
        return

    df_selected = find_nearest_frames(df, SNAPSHOT_TIMES)
    scheduler_obs = load_scheduler_observations(db_file)

    snapshot_data = build_all_snapshot_data(df_selected, scheduler_obs)
    plot_snapshot_grid(snapshot_data, output_file="cloud_tracking_snapshots_jul02.png")

if __name__ == "__main__":
    main()

Loaded 1376 candidate frames for 2025-07-02

Selected frames:
  requested 2025-07-02 02:17:07+00:00  ->  matched 2025-07-02 02:17:07.954000+00:00  (Δ=1.0 s)
  requested 2025-07-02 02:21:49+00:00  ->  matched 2025-07-02 02:21:49.722000+00:00  (Δ=0.7 s)
  requested 2025-07-02 03:21:24+00:00  ->  matched 2025-07-02 03:21:24.033000+00:00  (Δ=0.0 s)
  requested 2025-07-02 03:30:56+00:00  ->  matched 2025-07-02 03:30:56.908000+00:00  (Δ=0.9 s)
  requested 2025-07-02 03:43:50+00:00  ->  matched 2025-07-02 03:43:50.013000+00:00  (Δ=0.0 s)
  requested 2025-07-02 04:23:10+00:00  ->  matched 2025-07-02 04:23:10.216000+00:00  (Δ=0.2 s)
  requested 2025-07-02 04:30:32+00:00  ->  matched 2025-07-02 04:30:32.292000+00:00  (Δ=0.3 s)
  requested 2025-07-02 07:11:10+00:00  ->  matched 2025-07-02 07:11:10.252000+00:00  (Δ=0.3 s)
Using scheduler night 572 with 1139 observations

Saved figure: cloud_tracking_snapshots_jul02.png


In [11]:
#!/usr/bin/env python3

import io
import sqlite3
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from scipy.ndimage import gaussian_filter
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

# ======================
# PAPER-READY STYLE
# ======================
mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
    "axes.labelsize": 22,
    "axes.titlesize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 28,
    "lines.linewidth": 2.0,
    "axes.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

NSIDE_EXPECTED = 32
NEST = True
UNSEEN = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan
RUBIN_LAT = -30.244639
RUBIN_LON = -70.749417
RUBIN_HEIGHT_M = 2663.0
CBAR_VMIN = -0.2
CBAR_VMAX = 2.0
BIN_SIZE_KM = 1000
R_PROJECTION = 10000.0

# First two removed → now 6 snapshots
SNAPSHOT_TIMES = [
    "2025-07-02 03:28:24",
    "2025-07-02 03:30:56",
    "2025-07-02 03:43:50",
    "2025-07-02 04:23:10",
    "2025-07-02 04:30:32",
    "2025-07-02 07:11:10",
]

def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url

def fetch_map_resourcepath(url: str):
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    bio = io.BytesIO(data)
    with h5py.File(bio, "r") as f:
        if "clouds" not in f:
            raise RuntimeError(f"'clouds' not found in {url}")
        arr = np.array(f["clouds"]).ravel()
    return arr

def healpix_radec_to_ortho(mp, nside, nest=NEST,
                           observer_lat=RUBIN_LAT, observer_lon=RUBIN_LON,
                           observer_height_m=RUBIN_HEIGHT_M, obstime=None):
    if nside is None:
        nside = hp.get_nside(mp)
    npix = hp.nside2npix(nside)
    pix = np.arange(npix)
    theta_pix, phi_pix = hp.pix2ang(nside, pix, nest=nest)
    ra = np.degrees(phi_pix)
    dec = 90.0 - np.degrees(theta_pix)
    if obstime is None:
        obstime = Time.now().utc
    else:
        obstime = Time(obstime) if not isinstance(obstime, Time) else obstime
        obstime = obstime.utc
    location = EarthLocation(lat=observer_lat * u.deg,
                             lon=observer_lon * u.deg,
                             height=observer_height_m * u.m)
    sky = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")
    altaz = sky.transform_to(AltAz(obstime=obstime, location=location))
    alt_deg = altaz.alt.deg
    az_deg = altaz.az.deg % 360.0
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)
    return az_deg, alt_deg, vals

def radec_to_altaz_xy(ra_deg, dec_deg, obstime,
                      observer_lat=RUBIN_LAT, observer_lon=RUBIN_LON,
                      observer_height_m=RUBIN_HEIGHT_M):
    if not isinstance(obstime, Time):
        obstime = Time(obstime)
    obstime = obstime.utc

    location = EarthLocation(lat=observer_lat * u.deg,
                             lon=observer_lon * u.deg,
                             height=observer_height_m * u.m)
    sky = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg, frame="icrs")
    altaz = sky.transform_to(AltAz(obstime=obstime, location=location))

    alt_deg = altaz.alt.deg
    az_deg = altaz.az.deg % 360.0

    alt_rad = np.radians(alt_deg)
    az_rad = np.radians(az_deg)

    x_altaz = -np.cos(alt_rad) * np.sin(az_rad)
    y_altaz = np.cos(alt_rad) * np.cos(az_rad)
    z_altaz = np.sin(alt_rad)

    if z_altaz > 0.01:
        scale = R_PROJECTION / z_altaz
        x_km = x_altaz * scale
        y_km = y_altaz * scale
        return x_km, y_km, alt_deg, az_deg
    else:
        return None, None, alt_deg, az_deg

def process_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    az, alt, vals = healpix_radec_to_ortho(mp, NSIDE_EXPECTED, obstime=obstime)

    alt_rad = np.radians(alt)
    az_rad = np.radians(az)

    x_altaz = -np.cos(alt_rad) * np.sin(az_rad)
    y_altaz = np.cos(alt_rad) * np.cos(az_rad)
    z_altaz = np.sin(alt_rad)

    mask = z_altaz > 0.01
    scale = np.where(mask, R_PROJECTION / z_altaz, np.nan)

    x_flat = x_altaz * scale
    y_flat = y_altaz * scale
    vals_flat = np.where(mask, vals, np.nan)

    radius = np.sqrt(x_flat**2 + y_flat**2)
    crop_mask = radius <= 15000.0

    x_cropped = x_flat[crop_mask]
    y_cropped = y_flat[crop_mask]
    vals_cropped = vals_flat[crop_mask]

    valid_mask = ~np.isnan(vals_cropped)
    x_valid = x_cropped[valid_mask]
    y_valid = y_cropped[valid_mask]
    vals_valid = vals_cropped[valid_mask]

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)

    H_sum, _, _ = np.histogram2d(x_valid, y_valid, bins=[x_edges, y_edges], weights=vals_valid)
    H_count, _, _ = np.histogram2d(x_valid, y_valid, bins=[x_edges, y_edges])

    with np.errstate(divide='ignore', invalid='ignore'):
        H_avg = H_sum / H_count
        H_avg[H_count == 0] = np.nan

    H_avg = H_avg.T

    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    X_grid, Y_grid = np.meshgrid(x_centers, y_centers)

    R_grid = np.sqrt(X_grid**2 + Y_grid**2)
    H_avg[R_grid > 15000] = np.nan

    return H_avg, X_grid, Y_grid, x_edges, y_edges

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)

    g1_smooth = gaussian_filter(g1, sigma=sigma)
    g2_smooth = gaussian_filter(g2, sigma=sigma)

    mask1 = ~np.isnan(grid1) & (grid1 != 0)
    mask2 = ~np.isnan(grid2) & (grid2 != 0)

    best_corr = -np.inf
    best_dx, best_dy = 0, 0

    for dy in range(-search_range, search_range + 1):
        for dx in range(-search_range, search_range + 1):
            shifted = ndimage.shift(g2_smooth, (dy, dx), order=1, mode='constant', cval=0)
            shifted_mask = ndimage.shift(mask2.astype(float), (dy, dx), order=0, mode='constant', cval=0) > 0.5

            valid = mask1 & shifted_mask

            if np.sum(valid) < 100:
                continue

            v1 = g1_smooth[valid]
            v2 = shifted[valid]

            if len(v1) > 0 and np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0, 1]
                if corr > best_corr:
                    best_corr = corr
                    best_dx = dx
                    best_dy = dy

    return best_dx, best_dy, best_corr

def find_absolute_minimum(grid):
    valid_mask = ~np.isnan(grid)
    if not np.any(valid_mask):
        return 0, 0, np.nan

    min_idx = np.nanargmin(grid)
    y_idx, x_idx = np.unravel_index(min_idx, grid.shape)

    x_km = (x_idx - 15) * BIN_SIZE_KM
    y_km = (y_idx - 15) * BIN_SIZE_KM
    min_val = grid[y_idx, x_idx]

    return x_km, y_km, min_val

def predict_future_position(current_x, current_y, dx_pix, dy_pix):
    dx_km = dx_pix * BIN_SIZE_KM
    dy_km = dy_pix * BIN_SIZE_KM

    new_x = current_x - dx_km
    new_y = current_y - dy_km

    max_radius = 14000
    radius = np.sqrt(new_x**2 + new_y**2)
    if radius > max_radius:
        scale = max_radius / radius
        new_x *= scale
        new_y *= scale

    return new_x, new_y

def get_value_at_position(grid, x_km, y_km):
    x_idx = int((x_km / BIN_SIZE_KM) + 15)
    y_idx = int((y_km / BIN_SIZE_KM) + 15)

    if 0 <= x_idx < grid.shape[1] and 0 <= y_idx < grid.shape[0]:
        return grid[y_idx, x_idx]
    else:
        return np.nan

def load_night_data(csv_file, target_date="2025-07-02"):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', '').str.strip()

    time_col = 'time'
    url_col = 'lsst.sal.DREAM.logevent_largeFileObjectAvailable.url'

    df = df.dropna(subset=[url_col]).copy()
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce', utc=True)
    df = df.dropna(subset=[time_col]).copy()
    df = df[df[url_col].str.contains('.hdf5', case=False, na=False)].copy()
    df = df[df[url_col].str.contains('cloud_sys', case=False, na=False)].copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    start = pd.Timestamp(f"{target_date} 00:00:00", tz="UTC")
    end = pd.Timestamp(f"{target_date} 23:59:59", tz="UTC")
    df_night = df[(df[time_col] >= start) & (df[time_col] <= end)].copy()

    print(f"Loaded {len(df_night)} candidate frames for {target_date}")
    return df_night

def load_scheduler_observations(db_file):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]

    table_name = None
    for possible_name in ['observations', 'SummaryAllProps', 'Summary', 'obs']:
        if possible_name in tables:
            table_name = possible_name
            break
    if table_name is None:
        table_name = tables[0]

    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()

    night_col = None
    for col in obs_df.columns:
        if 'night' in col.lower():
            night_col = col
            break
    if night_col is None:
        raise RuntimeError("Could not find night column")

    obs_df['night'] = obs_df[night_col].astype(int)

    for req in ['observationStartMJD', 'fieldRA', 'fieldDec']:
        found = False
        for col in obs_df.columns:
            if req.lower() in col.lower():
                if col != req:
                    obs_df[req] = obs_df[col]
                found = True
                break
        if not found:
            raise RuntimeError(f"Could not find required column: {req}")

    best_night = obs_df.groupby('night').size().idxmax()
    night_obs = obs_df[obs_df['night'] == best_night].copy()
    night_obs = night_obs.sort_values('observationStartMJD').reset_index(drop=True)

    print(f"Using scheduler night {best_night} with {len(night_obs)} observations")
    return night_obs

def match_scheduler_to_frames(scheduler_obs, frame_times):
    n_frames = len(frame_times)
    frame_fractions = np.linspace(0, 1, n_frames)

    mjd_vals = scheduler_obs['observationStartMJD'].values
    mjd_min = mjd_vals.min()
    mjd_max = mjd_vals.max()
    mjd_range = mjd_max - mjd_min

    if mjd_range > 0:
        scheduler_fractions = (mjd_vals - mjd_min) / mjd_range
    else:
        scheduler_fractions = np.zeros(len(mjd_vals))

    matched_obs = []
    for frame_frac in frame_fractions:
        diffs = np.abs(scheduler_fractions - frame_frac)
        closest_idx = np.argmin(diffs)
        matched_obs.append(scheduler_obs.iloc[closest_idx])

    return matched_obs

def find_nearest_frames(df, requested_times):
    requested = pd.to_datetime(requested_times, utc=True)
    selected_rows = []

    for t in requested:
        idx = np.argmin(np.abs(df['time'] - t))
        row = df.iloc[idx].copy()
        row['requested_time'] = t
        row['time_offset_sec'] = abs((row['time'] - t).total_seconds())
        selected_rows.append(row)

    out = pd.DataFrame(selected_rows).drop_duplicates(subset=['time']).reset_index(drop=True)
    print("\nSelected frames:")
    for _, row in out.iterrows():
        print(f"  requested {row['requested_time']}  ->  matched {row['time']}  (Δ={row['time_offset_sec']:.1f} s)")
    return out

def build_all_snapshot_data(df_selected, scheduler_obs):
    all_grids = []
    all_metas = []
    X_grid = Y_grid = x_edges = y_edges = None

    for _, row in df_selected.iterrows():
        url = transform_url(row['lsst.sal.DREAM.logevent_largeFileObjectAvailable.url'])
        mp = fetch_map_resourcepath(url)
        obstime = row['time'].to_pydatetime()
        grid, Xg, Yg, xe, ye = process_to_grid(mp, obstime)

        all_grids.append(grid)
        all_metas.append({
            "url": url,
            "time": obstime,
            "requested_time": row['requested_time']
        })

        if X_grid is None:
            X_grid, Y_grid, x_edges, y_edges = Xg, Yg, xe, ye

    matched_scheduler = match_scheduler_to_frames(scheduler_obs, [m['time'] for m in all_metas])

    x_abs, y_abs, _ = find_absolute_minimum(all_grids[0])
    x_pred, y_pred = x_abs, y_abs

    abs_positions = [(x_abs, y_abs)]
    pred_positions = [(x_pred, y_pred)]
    motion_vectors = [(0, 0)]

    history_length = 3

    for i in range(1, len(all_grids)):
        x_abs, y_abs, _ = find_absolute_minimum(all_grids[i])
        abs_positions.append((x_abs, y_abs))

        if i >= history_length:
            dx_total, dy_total = 0, 0
            count = 0
            for j in range(1, history_length + 1):
                dx, dy, conf = compute_motion_with_correlation(
                    all_grids[i-j], all_grids[i-j+1], sigma=5.0, search_range=10
                )
                if conf > 0.5:
                    dx_total += dx
                    dy_total += dy
                    count += 1
            if count > 0:
                dx_avg = dx_total / count
                dy_avg = dy_total / count
            else:
                dx_avg, dy_avg = 0, 0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(
                all_grids[i-1], all_grids[i], sigma=5.0, search_range=10
            )

        motion_vectors.append((dx_avg, dy_avg))
        x_pred, y_pred = predict_future_position(x_pred, y_pred, dx_avg, dy_avg)
        pred_positions.append((x_pred, y_pred))

    return {
        "all_grids": all_grids,
        "all_metas": all_metas,
        "matched_scheduler": matched_scheduler,
        "abs_positions": abs_positions,
        "pred_positions": pred_positions,
        "motion_vectors": motion_vectors,
        "x_edges": x_edges,
        "y_edges": y_edges,
    }

def format_axis(ax, show_xlabel=False, show_ylabel=False):
    ax.set_xlim(-15000, 15000)
    ax.set_ylim(-15000, 15000)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.12)

    ticks = [-15000, 0, 15000]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)

    if show_xlabel:
        ax.set_xlabel(r"X ($10^3$ km)")
        ax.set_xticklabels(["−15", "0", "15"])
    else:
        ax.tick_params(labelbottom=False)

    if show_ylabel:
        ax.set_ylabel(r"Y ($10^3$ km)")
        ax.set_yticklabels(["−15", "0", "15"])
    else:
        ax.tick_params(labelleft=False)

def plot_snapshot_grid(snapshot_data, output_file_base="cloud_tracking_snapshots_jul02"):
    all_grids = snapshot_data["all_grids"][:6]
    all_metas = snapshot_data["all_metas"][:6]
    matched_scheduler = snapshot_data["matched_scheduler"][:6]
    abs_positions = snapshot_data["abs_positions"][:6]
    pred_positions = snapshot_data["pred_positions"][:6]
    motion_vectors = snapshot_data["motion_vectors"][:6]
    x_edges = snapshot_data["x_edges"] / 1000.0
    y_edges = snapshot_data["y_edges"] / 1000.0

    fig, axes = plt.subplots(
        2, 3,
        figsize=(20, 12),
        constrained_layout=False
    )
    axes = axes.flatten()

    im = None

    for i, ax in enumerate(axes):
        if i >= len(all_grids):
            ax.axis("off")
            continue

        grid = all_grids[i]

        im = ax.pcolormesh(
            x_edges, y_edges, grid,
            cmap="viridis",
            vmin=CBAR_VMIN,
            vmax=CBAR_VMAX,
            shading="flat"
        )

        theta = np.linspace(0, 2*np.pi, 300)
        ax.plot(15 * np.cos(theta), 15 * np.sin(theta), "k-", linewidth=1.4, alpha=0.45)

        x_abs, y_abs = abs_positions[i]
        x_abs_plot = x_abs / 1000.0
        y_abs_plot = y_abs / 1000.0
        val_abs = get_value_at_position(grid, x_abs, y_abs)
        ax.plot(
            x_abs_plot, y_abs_plot, "o",
            color="lime", markersize=12,
            markeredgewidth=1.8, markeredgecolor="white", zorder=10
        )

        x_pred, y_pred = pred_positions[i]
        x_pred_plot = x_pred / 1000.0
        y_pred_plot = y_pred / 1000.0
        val_pred = get_value_at_position(grid, x_pred, y_pred)
        ax.plot(
            x_pred_plot, y_pred_plot, "o",
            color="deepskyblue", markersize=12,
            markeredgewidth=1.8, markeredgecolor="white", zorder=10
        )

        obs = matched_scheduler[i]
        x_sched, y_sched, alt, az = radec_to_altaz_xy(
            obs["fieldRA"], obs["fieldDec"], all_metas[i]["time"]
        )
        val_sched = np.nan

        if x_sched is not None and y_sched is not None:
            val_sched = get_value_at_position(grid, x_sched, y_sched)
            ax.plot(
                x_sched / 1000.0, y_sched / 1000.0, "o",
                color="red", markersize=12,
                markeredgewidth=1.8, markeredgecolor="white", zorder=10
            )

        if i > 0:
            dx_pix, dy_pix = motion_vectors[i]
            dx_km = dx_pix * BIN_SIZE_KM
            dy_km = dy_pix * BIN_SIZE_KM
            mag = np.sqrt(dx_km**2 + dy_km**2)

            if mag > 300:
                arrow = FancyArrowPatch(
                    (x_pred_plot, y_pred_plot),
                    (x_pred_plot + dx_km / 1000.0, y_pred_plot + dy_km / 1000.0),
                    arrowstyle='-|>',
                    mutation_scale=22,
                    linewidth=2.4,
                    color='cyan',
                    alpha=0.95,
                    zorder=15
                )
                ax.add_patch(arrow)

        ax.plot(
            0, 0,
            marker="+",
            color="white",
            markersize=12,
            markeredgewidth=2.2,
            zorder=12
        )

        stats_label = (
            f"Abs: {val_abs:.2f}\n"
            f"Track: {val_pred:.2f}\n"
            f"Sched: {val_sched:.2f}"
            if not np.isnan(val_sched)
            else f"Abs: {val_abs:.2f}\nTrack: {val_pred:.2f}\nSched: off-grid"
        )

        ax.text(
            0.985, 0.985, stats_label,
            transform=ax.transAxes,
            fontsize=14,
            ha="right",
            va="top",
            color="white",
            weight="bold",
            bbox=dict(
                boxstyle="round,pad=0.20",
                facecolor="black",
                alpha=0.88,
                edgecolor="black",
                linewidth=0.8
            ),
            zorder=30
        )

        # TIMESTAMP IN TOP LEFT
        time_str = pd.Timestamp(all_metas[i]["time"]).strftime("%H:%M:%S")
        ax.text(
            0.02, 0.985, time_str,
            transform=ax.transAxes,
            fontsize=18,
            ha="left",
            va="top",
            color="white",
            weight="bold",
            bbox=dict(
                boxstyle="round,pad=0.18",
                facecolor="black",
                alpha=0.88,
                edgecolor="black",
                linewidth=0.8
            ),
            zorder=30
        )

        ax.set_xlim(-16, 16)
        ax.set_ylim(-16, 16)
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.08)

        ax.tick_params(
            labelsize=18,
            direction="in",
            top=True,
            right=True,
            pad=2
        )

        ax.set_xticks([-15, 0, 15])
        ax.set_yticks([-15, 0, 15])

        if i % 3 == 0:
            ax.set_ylabel(r"Y ($10^3$ km)", fontsize=22)
            ax.set_yticklabels(["−15", "0", "15"], fontsize=18)
        else:
            ax.tick_params(labelleft=False)

        if i >= 3:
            ax.set_xlabel(r"X ($10^3$ km)", fontsize=22)
            ax.set_xticklabels(["−15", "0", "15"], fontsize=18)
        else:
            ax.tick_params(labelbottom=False)

    plt.subplots_adjust(
        left=0.06,
        right=0.84,
        bottom=0.11,
        top=0.90,
        wspace=-0.235,
        hspace=0.00
    )

    cax = fig.add_axes([0.84, 0.18, 0.022, 0.64])
    cbar = fig.colorbar(im, cax=cax, extend="both", extendfrac=0.08)
    cbar.set_label("Cloud extinction (mag)", fontsize=20, labelpad=12)
    cbar.ax.tick_params(labelsize=16, direction="in")

    handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lime',
                   markeredgecolor='white', markersize=11, linestyle='None',
                   label='Absolute minimum'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='deepskyblue',
                   markeredgecolor='white', markersize=11, linestyle='None',
                   label='Motion tracking'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
                   markeredgecolor='white', markersize=11, linestyle='None',
                   label='Scheduler'),
        plt.Line2D([0], [0], marker='+', color='black', markersize=12,
                   markeredgewidth=2, linestyle='None', label='Zenith')
    ]

    fig.legend(
        handles=handles,
        loc='lower center',
        bbox_to_anchor=(0.43, 0),
        ncol=4,
        fontsize=16,
        frameon=True,
        edgecolor="black"
    )

    fig.suptitle(
        "Cloud Motion Tracking vs Scheduler: Selected DREAM Snapshots from 2025-07-02",
        fontsize=26,
        weight='bold',
        y=0.965
    )

    plt.savefig(f"{output_file_base}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{output_file_base}.pdf", dpi=300, bbox_inches="tight")
    plt.close()

    print(f"\nSaved figure: {output_file_base}.png")
    print(f"Saved figure: {output_file_base}.pdf")

def main():
    csv_file = "feb5_data.csv"
    db_file = "baseline_v5.1.0_10yrs.db"

    df = load_night_data(csv_file, target_date="2025-07-02")
    if len(df) == 0:
        print("No matching cloud frames found.")
        return

    df_selected = find_nearest_frames(df, SNAPSHOT_TIMES)
    scheduler_obs = load_scheduler_observations(db_file)

    snapshot_data = build_all_snapshot_data(df_selected, scheduler_obs)
    plot_snapshot_grid(snapshot_data, output_file_base="cloud_tracking_snapshots_jul02_pretty")

if __name__ == "__main__":
    main()

Loaded 1376 candidate frames for 2025-07-02

Selected frames:
  requested 2025-07-02 03:28:24+00:00  ->  matched 2025-07-02 03:28:16.312000+00:00  (Δ=7.7 s)
  requested 2025-07-02 03:30:56+00:00  ->  matched 2025-07-02 03:30:56.908000+00:00  (Δ=0.9 s)
  requested 2025-07-02 03:43:50+00:00  ->  matched 2025-07-02 03:43:50.013000+00:00  (Δ=0.0 s)
  requested 2025-07-02 04:23:10+00:00  ->  matched 2025-07-02 04:23:10.216000+00:00  (Δ=0.2 s)
  requested 2025-07-02 04:30:32+00:00  ->  matched 2025-07-02 04:30:32.292000+00:00  (Δ=0.3 s)
  requested 2025-07-02 07:11:10+00:00  ->  matched 2025-07-02 07:11:10.252000+00:00  (Δ=0.3 s)
Using scheduler night 572 with 1139 observations

Saved figure: cloud_tracking_snapshots_jul02_pretty.png
Saved figure: cloud_tracking_snapshots_jul02_pretty.pdf


In [13]:
#!/usr/bin/env python3
"""
Run full DREAM pipeline for a single night (2025-09-12) starting from frame 210.
Produce the standard metrics plot (histograms + timelines) with publication-ready
styling (STIXGeneral, large fonts). Save plot as PDF/PNG and save metrics data.
"""

import io
import os
import sqlite3
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import gaussian_filter
from scipy import ndimage, stats
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from lsst.summit.utils import ConsDbClient

import warnings
warnings.filterwarnings("ignore")

# ======================
# PUBLICATION-READY STYLE (as used in snapshot script)
# ======================
mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
    "axes.labelsize": 22,
    "axes.titlesize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 28,
    "lines.linewidth": 2.0,
    "axes.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

# ======================
# CONSTANTS (same as original pipeline)
# ======================
NSIDE_EXPECTED = 32
NEST = True
UNSEEN = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan
RUBIN_LAT = -30.244639
RUBIN_LON = -70.749417
RUBIN_HEIGHT_M = 2663.0
BIN_SIZE_KM = 1000
R_PROJECTION = 10000.0
MIN_ALT_DEG = 15.0
MOON_EXCLUSION_DEG = 30.0
W_CLOUD = 0.50
W_SLEW = 0.25
W_ZENITH = 0.25
MAX_EXTINCTION_NORM = 2.0
MAX_SLEW_NORM = 60.0
EXPOSURE_TIME = 30.0
READOUT_TIME = 2.0
SLOT_TIME = EXPOSURE_TIME + READOUT_TIME
MAX_SLEW_SPEED_ALT = 3.5
MAX_SLEW_SPEED_AZ = 7.0
SLEW_SETTLE_TIME = 2.0
PHOTON_FLUX_MAG20_ZENITH = 100.0
RUBIN_EFFECTIVE_AREA = np.pi * (6.4 / 2) ** 2
RUBIN_QUANTUM_EFFICIENCY = 0.8
MAX_SIGMA_MAG = 0.3
MAX_FLAG_VALUE = 0
PATCHINESS_SAMPLE_N = 10
PATCHINESS_LOW_FRAC = 0.10
PATCHINESS_HIGH_FRAC = 0.90
PATCHINESS_VAR_THRESH = 0.05
ALPHA_SEEING = 0.6
BETA_SEEING = 0.2
FWHM_CONSTANT = 2 * np.sqrt(2 * np.log(2))
EFFECTIVE_WAVELENGTHS = {'u':366.3, 'g':481.0, 'r':622.2, 'i':754.5, 'z':869.1, 'y':971.0}
URL_COL = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"

# ======================
# UTILITY FUNCTIONS (from pipeline)
# ======================
def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url

def fetch_sys_map(url):
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    bio = io.BytesIO(data)
    with h5py.File(bio, "r") as f:
        clouds = np.array(f["clouds"]).ravel()
        sigma = np.array(f["sigma"]).ravel()
        flags = np.array(f["flags"]).ravel()
        mask_visible = np.array(f["mask_visible"]).ravel()
        nobs = np.array(f["nobs"]).ravel()
    bad = (~mask_visible) | (nobs == 0) | (flags > MAX_FLAG_VALUE) | (sigma > MAX_SIGMA_MAG) | (~np.isfinite(clouds))
    clouds[bad] = np.nan
    sigma[bad] = np.nan
    return clouds, sigma

def _make_location():
    return EarthLocation(lat=RUBIN_LAT*u.deg, lon=RUBIN_LON*u.deg, height=RUBIN_HEIGHT_M*u.m)

def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc

def radec_to_altaz_xy(ra_deg, dec_deg, obstime):
    t = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az = aa.az.deg % 360.0
    if alt < MIN_ALT_DEG:
        return None, None, alt, az
    alt_r = np.radians(alt)
    az_r = np.radians(az)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return x, y, alt, az

def altaz_to_xy(alt_deg, az_deg):
    if alt_deg < MIN_ALT_DEG:
        return None, None
    alt_r = np.radians(alt_deg)
    az_r = np.radians(az_deg)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return x, y

def xy_to_altaz(x_km, y_km):
    r = np.sqrt(x_km**2 + y_km**2)
    alt = np.degrees(np.arctan2(R_PROJECTION, r))
    az = np.degrees(np.arctan2(-x_km, y_km)) % 360.0
    return alt, az

def get_moon_altaz(obstime):
    t = _ensure_time(obstime)
    loc = _make_location()
    moon = get_body("moon", t, loc)
    aa = moon.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)

def angular_separation(alt1, az1, alt2, az2):
    a1, a2 = np.radians(alt1), np.radians(alt2)
    z1, z2 = np.radians(az1), np.radians(az2)
    cos_sep = np.sin(a1)*np.sin(a2) + np.cos(a1)*np.cos(a2)*np.cos(z1-z2)
    cos_sep = np.clip(cos_sep, -1.0, 1.0)
    return np.degrees(np.arccos(cos_sep))

def is_moon_safe(alt, az, moon_alt, moon_az):
    if alt < MIN_ALT_DEG:
        return False
    return angular_separation(alt, az, moon_alt, moon_az) >= MOON_EXCLUSION_DEG

def healpix_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    npix = hp.nside2npix(NSIDE_EXPECTED)
    pix = np.arange(npix)
    theta, phi = hp.pix2ang(NSIDE_EXPECTED, pix, nest=NEST)
    ra = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)
    t = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az = aa.az.deg % 360.0
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)

    alt_r = np.radians(alt)
    az_r = np.radians(az)
    above = alt > MIN_ALT_DEG
    scale = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf = -np.cos(alt_r) * np.sin(az_r) * scale
    yf =  np.cos(alt_r) * np.cos(az_r) * scale
    vf = np.where(above, vals, np.nan)

    r = np.sqrt(xf**2 + yf**2)
    c = r <= 15000.0
    ok = ~np.isnan(vf[c])

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)
    Hs, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges], weights=vf[c][ok])
    Hc, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15000] = np.nan
    return H, Xg, Yg, x_edges, y_edges

def calculate_slew_time(alt1, az1, alt2, az2):
    da = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_SPEED_ALT, daz / MAX_SLEW_SPEED_AZ) + SLEW_SETTLE_TIME

def compute_photon_collection(ext_mag):
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    rate = PHOTON_FLUX_MAG20_ZENITH * RUBIN_EFFECTIVE_AREA * RUBIN_QUANTUM_EFFICIENCY * flux
    return rate * EXPOSURE_TIME

def photons_to_magnitude(photons):
    if np.isnan(photons) or photons <= 0:
        return np.nan
    snr = photons / (5 * np.sqrt(photons))
    return 20.0 - 2.5 * np.log10(snr) if snr > 0 else np.nan

def get_value_at_position(grid, x_km, y_km):
    xi = int(round((x_km / BIN_SIZE_KM) + 15))
    yi = int(round((y_km / BIN_SIZE_KM) + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan

def find_absolute_minimum(grid, moon_alt=None, moon_az=None):
    work = grid.copy()
    if moon_alt is not None and moon_az is not None:
        ny, nx = work.shape
        for yi in range(ny):
            for xi in range(nx):
                if np.isnan(work[yi, xi]):
                    continue
                x_km = (xi - 15) * BIN_SIZE_KM
                y_km = (yi - 15) * BIN_SIZE_KM
                alt, az = xy_to_altaz(x_km, y_km)
                if not is_moon_safe(alt, az, moon_alt, moon_az):
                    work[yi, xi] = np.nan
    if not np.any(~np.isnan(work)):
        work = grid.copy()
    if not np.any(~np.isnan(work)):
        return 0, 0, np.nan
    idx = np.nanargmin(work)
    yi, xi = np.unravel_index(idx, work.shape)
    x = (xi - 15) * BIN_SIZE_KM
    y = (yi - 15) * BIN_SIZE_KM
    return x, y, grid[yi, xi]

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)
    g1s = gaussian_filter(g1, sigma=sigma)
    g2s = gaussian_filter(g2, sigma=sigma)
    m1 = ~np.isnan(grid1) & (grid1 != 0)
    m2 = ~np.isnan(grid2) & (grid2 != 0)
    best_corr, best_dx, best_dy = -np.inf, 0, 0
    for dy in range(-search_range, search_range+1):
        for dx in range(-search_range, search_range+1):
            sh = ndimage.shift(g2s, (dy, dx), order=1, mode='constant', cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode='constant', cval=0) > 0.5
            valid = m1 & sm
            if np.sum(valid) < 100:
                continue
            v1, v2 = g1s[valid], sh[valid]
            if np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0,1]
                if corr > best_corr:
                    best_corr, best_dx, best_dy = corr, dx, dy
    return best_dx, best_dy, best_corr

def score_candidate(x_km, y_km, ext_val, prev_alt, prev_az, moon_alt, moon_az):
    if np.isnan(ext_val):
        return np.inf
    alt, az = xy_to_altaz(x_km, y_km)
    if alt < MIN_ALT_DEG:
        return np.inf
    if not is_moon_safe(alt, az, moon_alt, moon_az):
        return np.inf
    cloud_norm = np.clip(ext_val / MAX_EXTINCTION_NORM, 0.0, 1.0)
    if prev_alt is not None:
        slew = calculate_slew_time(prev_alt, prev_az, alt, az)
    else:
        slew = 0.0
    slew_norm = np.clip(slew / MAX_SLEW_NORM, 0.0, 1.0)
    zenith_norm = (90.0 - alt) / 90.0
    return W_CLOUD * cloud_norm + W_SLEW * slew_norm + W_ZENITH * zenith_norm

def predict_future_position(cx, cy, dx_pix, dy_pix, current_grid, prev_alt, prev_az, moon_alt, moon_az):
    nx = cx - dx_pix * BIN_SIZE_KM
    ny = cy - dy_pix * BIN_SIZE_KM
    r = np.sqrt(nx**2 + ny**2)
    if r > 14000:
        nx *= 14000 / r
        ny *= 14000 / r
    # sample 9 positions around prediction
    candidates = [(nx, ny)]
    for dy in [-1,0,1]:
        for dx in [-1,0,1]:
            if dx==0 and dy==0: continue
            candidates.append((nx + dx*BIN_SIZE_KM, ny + dy*BIN_SIZE_KM))
    best_score = np.inf
    best_x, best_y = nx, ny
    for cx_, cy_ in candidates:
        ext = get_value_at_position(current_grid, cx_, cy_)
        s = score_candidate(cx_, cy_, ext, prev_alt, prev_az, moon_alt, moon_az)
        if s < best_score:
            best_score = s
            best_x, best_y = cx_, cy_
    if best_score == np.inf:
        best_x, best_y, _ = find_absolute_minimum(current_grid, moon_alt, moon_az)
    return best_x, best_y

def load_night_sys_frames(csv_file, target_date, start_frame=0):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', '').str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains('.hdf5', case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains('cloud_sys', case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    start = pd.Timestamp(f"{target_date} 00:00:00", tz="UTC")
    end   = pd.Timestamp(f"{target_date} 23:59:59", tz="UTC")
    df_night = df[(df[TIME_COL] >= start) & (df[TIME_COL] <= end)].copy()
    if len(df_night) <= start_frame:
        raise ValueError(f"Only {len(df_night)} frames, cannot start at index {start_frame}")
    df_night = df_night.iloc[start_frame:].reset_index(drop=True)
    print(f"Loaded {len(df_night)} frames for {target_date} (starting at original frame {start_frame})")
    return df_night

def load_scheduler_observations(db_file):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    table_name = next((n for n in ['observations','SummaryAllProps','Summary','obs'] if n in tables), tables[0])
    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    night_col = next((c for c in obs_df.columns if 'night' in c.lower()), None)
    if night_col is None:
        raise RuntimeError("No night column")
    obs_df['night'] = obs_df[night_col].astype(int)
    for req in ['observationStartMJD','fieldRA','fieldDec']:
        for col in obs_df.columns:
            if req.lower() in col.lower():
                obs_df[req] = obs_df[col]
                break
    best_night = obs_df.groupby('night').size().idxmax()
    night_obs = obs_df[obs_df['night'] == best_night].copy()
    night_obs = night_obs.sort_values('observationStartMJD').reset_index(drop=True)
    print(f"Scheduler night {best_night} with {len(night_obs)} obs")
    return night_obs

def match_scheduler_to_frames(scheduler_obs, frame_times):
    n = len(frame_times)
    fracs = np.linspace(0, 1, n)
    mjd = scheduler_obs['observationStartMJD'].values
    mjd_range = mjd.max() - mjd.min()
    s_fracs = (mjd - mjd.min()) / mjd_range if mjd_range > 0 else np.zeros(len(mjd))
    matched = [scheduler_obs.iloc[np.argmin(np.abs(s_fracs - f))] for f in fracs]
    return matched

def build_consdb_df(start_night, instrument="lsstcam", consdb_url="http://consdb-pq.consdb:8080/consdb", h5_file=None):
    if h5_file and os.path.exists(h5_file):
        print(f"Loading ConsDB from {h5_file}")
        df = pd.read_hdf(h5_file, key="visits")
        df.columns = df.columns.str.strip()
        if "obs_midpoint" not in df.columns:
            t0 = pd.to_datetime(df["obs_start"], utc=True, errors="coerce")
            t1 = pd.to_datetime(df["obs_end"], utc=True, errors="coerce")
            df["obs_midpoint"] = t0 + (t1 - t0)/2
        for want, cands in {"ra":["s_ra","ra_v1","ra_ql","ra"], "dec":["s_dec","dec_v1","dec_ql","dec"]}.items():
            if want not in df.columns:
                for c in cands:
                    if c in df.columns:
                        df[want] = df[c]; break
        if "zp_consdb" not in df.columns and "zero_point_median" in df.columns:
            df["zp_consdb"] = pd.to_numeric(df["zero_point_median"], errors="coerce")
        df = df[df["ra"].notna() & df["dec"].notna()].reset_index(drop=True)
        print(f"ConsDB loaded: {len(df)} rows, zp non-null: {df['zp_consdb'].notna().sum()}")
        return df
    # live API – simplified, but you can keep your full implementation
    print("Live ConsDB query not implemented; returning empty DataFrame.")
    return pd.DataFrame()

def compute_all_positions(all_grids, all_metas):
    t0 = all_metas[0]["time"]
    m_alt0, m_az0 = get_moon_altaz(t0)
    x0, y0, _ = find_absolute_minimum(all_grids[0], m_alt0, m_az0)
    abs_pos = [(x0, y0)]
    pred_pos = [(x0, y0)]
    motion_v = [(0,0)]
    moon_pos = [(m_alt0, m_az0)]
    xp, yp = x0, y0
    prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    HL = 3
    for i in range(1, len(all_grids)):
        m_alt, m_az = get_moon_altaz(all_metas[i]["time"])
        moon_pos.append((m_alt, m_az))
        xa, ya, _ = find_absolute_minimum(all_grids[i], m_alt, m_az)
        abs_pos.append((xa, ya))
        if i >= HL:
            dxs, dys = [], []
            for j in range(1, HL+1):
                dx, dy, conf = compute_motion_with_correlation(all_grids[i-j], all_grids[i-j+1])
                if conf > 0.5:
                    dxs.append(dx); dys.append(dy)
            dx_avg = np.mean(dxs) if dxs else 0.0
            dy_avg = np.mean(dys) if dys else 0.0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(all_grids[i-1], all_grids[i])
        motion_v.append((dx_avg, dy_avg))
        xp, yp = predict_future_position(xp, yp, dx_avg, dy_avg, all_grids[i],
                                         prev_alt_p, prev_az_p, m_alt, m_az)
        pred_pos.append((xp, yp))
        prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    return abs_pos, pred_pos, motion_v, moon_pos

def compute_metrics(all_grids, all_sigma, all_metas, abs_pos, pred_pos, moon_pos, matched_sched):
    metrics = {s: {"photons":[], "sigma":[], "zenith_angles":[], "extinctions":[],
                   "slew_times":[], "expose_times":[], "frame_indices":[],
                   "alt_az":[], "xy_km":[]}
               for s in ("absolute","motion","scheduler")}
    prev_alt = {s:None for s in metrics}
    prev_az  = {s:None for s in metrics}
    for i in range(len(all_grids)):
        grid = all_grids[i]
        sgrid = all_sigma[i]
        m_alt, m_az = moon_pos[i]
        # absolute
        xa, ya = abs_pos[i]
        alt_a, az_a = xy_to_altaz(xa, ya)
        if alt_a >= MIN_ALT_DEG and is_moon_safe(alt_a, az_a, m_alt, m_az):
            ext = get_value_at_position(grid, xa, ya)
            sig = get_value_at_position(sgrid, xa, ya)
            za = 90.0 - alt_a
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["absolute"], prev_az["absolute"], alt_a, az_a) if prev_alt["absolute"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["absolute"]["photons"].append(collected)
                metrics["absolute"]["sigma"].append(sig)
                metrics["absolute"]["zenith_angles"].append(za)
                metrics["absolute"]["extinctions"].append(ext)
                metrics["absolute"]["slew_times"].append(slew)
                metrics["absolute"]["expose_times"].append(expose_t)
                metrics["absolute"]["frame_indices"].append(i)
                metrics["absolute"]["alt_az"].append((alt_a, az_a))
                metrics["absolute"]["xy_km"].append((xa, ya))
                prev_alt["absolute"], prev_az["absolute"] = alt_a, az_a
        # motion
        xp, yp = pred_pos[i]
        alt_p, az_p = xy_to_altaz(xp, yp)
        if alt_p >= MIN_ALT_DEG and is_moon_safe(alt_p, az_p, m_alt, m_az):
            ext = get_value_at_position(grid, xp, yp)
            sig = get_value_at_position(sgrid, xp, yp)
            za = 90.0 - alt_p
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["motion"], prev_az["motion"], alt_p, az_p) if prev_alt["motion"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["motion"]["photons"].append(collected)
                metrics["motion"]["sigma"].append(sig)
                metrics["motion"]["zenith_angles"].append(za)
                metrics["motion"]["extinctions"].append(ext)
                metrics["motion"]["slew_times"].append(slew)
                metrics["motion"]["expose_times"].append(expose_t)
                metrics["motion"]["frame_indices"].append(i)
                metrics["motion"]["alt_az"].append((alt_p, az_p))
                metrics["motion"]["xy_km"].append((xp, yp))
                prev_alt["motion"], prev_az["motion"] = alt_p, az_p
        # scheduler
        obs = matched_sched[i]
        xs, ys, alt_s, az_s = radec_to_altaz_xy(obs['fieldRA'], obs['fieldDec'], all_metas[i]["time"])
        if xs is not None:
            ext = get_value_at_position(grid, xs, ys)
            sig = get_value_at_position(sgrid, xs, ys)
            za = 90.0 - alt_s
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["scheduler"], prev_az["scheduler"], alt_s, az_s) if prev_alt["scheduler"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["scheduler"]["photons"].append(collected)
                metrics["scheduler"]["sigma"].append(sig)
                metrics["scheduler"]["zenith_angles"].append(za)
                metrics["scheduler"]["extinctions"].append(ext)
                metrics["scheduler"]["slew_times"].append(slew)
                metrics["scheduler"]["expose_times"].append(expose_t)
                metrics["scheduler"]["frame_indices"].append(i)
                metrics["scheduler"]["alt_az"].append((alt_s, az_s))
                metrics["scheduler"]["xy_km"].append((xs, ys))
                prev_alt["scheduler"], prev_az["scheduler"] = alt_s, az_s
    return metrics

def apply_atm_seeing_correction(metrics, all_metas, consdb_df):
    # Simplified: if consdb_df empty, just copy raw
    for s in metrics:
        n = len(metrics[s]["photons"])
        metrics[s]["photons_atm_corrected"] = metrics[s]["photons"][:]
        metrics[s]["eff_time_scale"] = [np.nan]*n
        metrics[s]["seeing_fwhm"] = [np.nan]*n
        metrics[s]["airmass_consdb"] = [np.nan]*n
    if consdb_df.empty:
        print("No ConsDB data, skipping corrections.")
    else:
        print("ConsDB corrections not fully implemented; using raw photons.")
    return metrics

def create_metrics_plot(metrics, output_pdf="metrics_20250912.pdf", output_png="metrics_20250912.png"):
    strategies = ("absolute", "motion", "scheduler")
    colors = ("green", "blue", "red")
    labels = ("Absolute Min", "Motion Tracking", "Scheduler")

    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(4, 3, hspace=0.40, wspace=0.32)

    # Row 0: Zenith angle, slew time, sigma
    ax = fig.add_subplot(gs[0,0])
    for s,c,l in zip(strategies, colors, labels):
        if metrics[s]["zenith_angles"]:
            ax.hist(metrics[s]["zenith_angles"], bins=20, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Zenith Angle (°)"); ax.set_ylabel("Count")
    ax.set_title("Zenith Angle Distribution", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0,1])
    for s,c,l in zip(strategies, colors, labels):
        slew = metrics[s]["slew_times"][1:]
        if slew:
            ax.hist(slew, bins=30, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Slew Time (s)"); ax.set_ylabel("Count")
    ax.set_title("Slew Time Distribution", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0,2])
    for s,c,l in zip(strategies, colors, labels):
        sigs = [x for x in metrics[s]["sigma"] if not np.isnan(x)]
        if sigs:
            ax.hist(sigs, bins=30, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.axvline(MAX_SIGMA_MAG, color='k', linestyle='--', lw=1.5, label=f"Mask threshold ({MAX_SIGMA_MAG})")
    ax.set_xlabel("Pixel σ uncertainty (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Uncertainty", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    # Row 1: Photon timeline (raw only, but we can show corrected if available)
    ax = fig.add_subplot(gs[1,:])
    for s,c,l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        ph = metrics[s]["photons"]
        if fi:
            ax.plot(fi, ph, color=c, alpha=0.7, linewidth=1.5, label=l)
            if "photons_atm_corrected" in metrics[s] and metrics[s]["photons_atm_corrected"]:
                phc = metrics[s]["photons_atm_corrected"]
                ax.plot(fi, phc, color=c, alpha=0.9, linestyle='--', linewidth=1.8, label=f"{l} (corr)")
    ax.set_xlabel("Frame Number (Time →)"); ax.set_ylabel("Slew-gated photons per slot")
    ax.set_title("Photon Collection Over the Night", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

    # Row 2: Total photons, median depth, efficiency
    ax = fig.add_subplot(gs[2,0])
    x_pos = np.arange(len(strategies))
    width = 0.35
    totals_raw = [np.sum(metrics[s]["photons"]) if metrics[s]["photons"] else 0 for s in strategies]
    totals_cor = [np.nansum(metrics[s].get("photons_atm_corrected", [np.nan])) for s in strategies]
    ax.bar(x_pos - width/2, totals_raw, width, color=colors, alpha=0.55, edgecolor='black', linewidth=1.2, label='raw')
    ax.bar(x_pos + width/2, totals_cor, width, color=colors, alpha=0.9, edgecolor='black', linewidth=1.2, hatch='//', label='atm-corrected')
    baseline = totals_raw[0] if totals_raw[0]>0 else 1
    for i, t in enumerate(totals_raw):
        ax.text(i - width/2, t, f"{((t-baseline)/baseline)*100:+.1f}%", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_xticks(x_pos); ax.set_xticklabels(labels)
    ax.set_ylabel("Total Photons"); ax.set_title("Total Night Photon Collection", weight="bold")
    ax.legend(fontsize=10); ax.grid(alpha=0.3, axis='y')
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

    ax = fig.add_subplot(gs[2,1])
    depths = []
    for s in strategies:
        mags = [photons_to_magnitude(p) for p in metrics[s]["photons"]]
        mags = [m for m in mags if not np.isnan(m)]
        depths.append(np.median(mags) if mags else 0)
    bars = ax.bar(labels, depths, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for bar, val in zip(bars, depths):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.2f}", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_ylabel("Median 5σ Depth (mag)"); ax.set_title("Survey Depth (slew-scaled)", weight="bold")
    ax.grid(alpha=0.3, axis='y'); ax.invert_yaxis()

    ax = fig.add_subplot(gs[2,2])
    effs = []
    for s in strategies:
        et = np.array(metrics[s]["expose_times"])
        slots = len(et) * SLOT_TIME
        effs.append(np.sum(et) / slots * 100 if slots>0 else 0)
    bars = ax.bar(labels, effs, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for bar, val in zip(bars, effs):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.1f}%", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_ylabel("Shutter-open efficiency (%)"); ax.set_title("Observation Efficiency", weight="bold")
    ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0,105)

    # Row 3: Seeing and eff_time_scale (if available)
    ax = fig.add_subplot(gs[3,0])
    for s,c,l in zip(strategies, colors, labels):
        sf = [v for v in metrics[s].get("seeing_fwhm", []) if np.isfinite(v)]
        if sf:
            ax.hist(sf, bins=20, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Seeing FWHM (arcsec)"); ax.set_ylabel("Count")
    ax.set_title("Seeing Distribution (ConsDB)", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[3,1])
    for s,c,l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        ets = metrics[s].get("eff_time_scale", [])
        if fi and len(ets)==len(fi):
            ax.plot(fi, ets, color=c, alpha=0.8, linewidth=1.5, label=l)
    ax.axhline(1.0, color='k', ls='--', lw=1.2, label='ideal (1.0)')
    ax.set_xlabel("Frame Number"); ax.set_ylabel("eff_time_scale")
    ax.set_title("Effective-Time Scale Factor (ConsDB)", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[3,2])
    for s,c,l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        am = metrics[s].get("airmass_consdb", [])
        if fi and len(am)==len(fi):
            am_f = [v if np.isfinite(v) else np.nan for v in am]
            ax.plot(fi, am_f, color=c, alpha=0.8, linewidth=1.5, label=l)
    ax.set_xlabel("Frame Number"); ax.set_ylabel("Airmass")
    ax.set_title("Airmass Over Night (ConsDB)", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    plt.suptitle("Rubin Observatory Pointing Strategy Comparison\n(σ-masked, slew-gated, moon-avoided, zenith-weighted, atm+seeing corrected)",
                 fontsize=16, weight="bold", y=0.995)
    plt.savefig(output_pdf, dpi=150, bbox_inches="tight")
    plt.savefig(output_png, dpi=150, bbox_inches="tight")
    print(f"Metrics plot saved: {output_pdf} and {output_png}")
    plt.close()

def save_metrics_data(metrics, output_csv="metrics_data_20250912.csv"):
    rows = []
    for strategy in ["absolute","motion","scheduler"]:
        n = len(metrics[strategy]["photons"])
        for i in range(n):
            row = {
                "strategy": strategy,
                "frame_idx": metrics[strategy]["frame_indices"][i],
                "photons_raw": metrics[strategy]["photons"][i],
                "extinction": metrics[strategy]["extinctions"][i],
                "sigma": metrics[strategy]["sigma"][i],
                "zenith_angle": metrics[strategy]["zenith_angles"][i],
                "slew_time": metrics[strategy]["slew_times"][i],
                "expose_time": metrics[strategy]["expose_times"][i],
            }
            if "photons_atm_corrected" in metrics[strategy]:
                row["photons_corrected"] = metrics[strategy]["photons_atm_corrected"][i]
                row["eff_time_scale"] = metrics[strategy]["eff_time_scale"][i]
                row["seeing_fwhm"] = metrics[strategy]["seeing_fwhm"][i]
                row["airmass_consdb"] = metrics[strategy]["airmass_consdb"][i]
            rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Metrics data saved to {output_csv}")

# ======================
# MAIN
# ======================
def main():
    csv_file = "feb5_data.csv"
    db_file = "baseline_v5.1.0_10yrs.db"
    target_date = "2025-09-12"
    start_frame = 210
    consdb_h5 = None   # optional: path to pre‑saved ConsDB HDF5

    # Load frames
    df_frames = load_night_sys_frames(csv_file, target_date, start_frame)
    if len(df_frames) < 5:
        print("Not enough frames, aborting.")
        return

    # Load scheduler
    scheduler_obs = load_scheduler_observations(db_file)

    # Load all grids, sigma, metas
    print("Loading all frames...")
    all_grids = []
    all_sigma = []
    all_metas = []
    for _, row in df_frames.iterrows():
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            obstime = row[TIME_COL].to_pydatetime()
            grid, _, _, _, _ = healpix_to_grid(clouds, obstime)
            # sigma grid (same projection)
            sig_grid, _, _, _, _ = healpix_to_grid(sigma, obstime)
            all_grids.append(grid)
            all_sigma.append(sig_grid)
            all_metas.append({"time": obstime})
        except Exception as e:
            print(f"Failed to load frame: {e}")
    print(f"Loaded {len(all_grids)} valid frames.")

    # Match scheduler
    matched_sched = match_scheduler_to_frames(scheduler_obs, [m["time"] for m in all_metas])

    # Motion tracking
    print("Running cloud motion tracking...")
    abs_pos, pred_pos, motion_v, moon_pos = compute_all_positions(all_grids, all_metas)

    # Compute metrics
    print("Computing metrics...")
    metrics = compute_metrics(all_grids, all_sigma, all_metas, abs_pos, pred_pos, moon_pos, matched_sched)

    # Apply ConsDB corrections (if available)
    consdb_df = build_consdb_df("2025-05-01", h5_file=consdb_h5)
    metrics = apply_atm_seeing_correction(metrics, all_metas, consdb_df)

    # Create plots
    create_metrics_plot(metrics,
                        output_pdf="metrics_20250912.pdf",
                        output_png="metrics_20250912.png")

    # Save data
    save_metrics_data(metrics, output_csv="metrics_data_20250912.csv")

    # Print summary
    print("\n=== Summary ===")
    for s in ["absolute","motion","scheduler"]:
        ph = np.array(metrics[s]["photons"])
        if len(ph):
            print(f"{s:10s}: {len(ph)} slots, total photons = {np.sum(ph):.3e}, "
                  f"median depth = {np.median([photons_to_magnitude(p) for p in ph if not np.isnan(p)]):.2f} mag")

if __name__ == "__main__":
    main()

Loaded 1002 frames for 2025-09-12 (starting at original frame 210)
Scheduler night 572 with 1139 obs
Loading all frames...
Loaded 1002 valid frames.
Running cloud motion tracking...
Computing metrics...
Live ConsDB query not implemented; returning empty DataFrame.
No ConsDB data, skipping corrections.
Metrics plot saved: metrics_20250912.pdf and metrics_20250912.png
Metrics data saved to metrics_data_20250912.csv

=== Summary ===
absolute  : 1002 slots, total photons = 5.828e+07, median depth = 15.73 mag
motion    : 1002 slots, total photons = 6.715e+07, median depth = 15.70 mag
scheduler : 77 slots, total photons = 4.395e+06, median depth = 15.81 mag


In [14]:
#!/usr/bin/env python3
"""
Run DREAM pipeline for a list of nights (in reverse order) and produce
metrics plots without the bottom row (seeing, eff_time_scale, airmass).
No ConsDB corrections – raw slew‑gated photons only.
"""

import io
import os
import sqlite3
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyArrowPatch
from matplotlib.animation import FuncAnimation, writers
from scipy.ndimage import gaussian_filter
from scipy import ndimage
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
import warnings
warnings.filterwarnings("ignore")

# ======================
# PUBLICATION-READY STYLE (same as used for 6‑panel snapshots)
# ======================
mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
    "axes.labelsize": 22,
    "axes.titlesize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 28,
    "lines.linewidth": 2.0,
    "axes.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

# ======================
# CONSTANTS (from original pipeline)
# ======================
NSIDE_EXPECTED = 32
NEST = True
UNSEEN = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan
RUBIN_LAT = -30.244639
RUBIN_LON = -70.749417
RUBIN_HEIGHT_M = 2663.0
BIN_SIZE_KM = 1000
R_PROJECTION = 10000.0
MIN_ALT_DEG = 15.0
MOON_EXCLUSION_DEG = 30.0
W_CLOUD = 0.50
W_SLEW = 0.25
W_ZENITH = 0.25
MAX_EXTINCTION_NORM = 2.0
MAX_SLEW_NORM = 60.0
EXPOSURE_TIME = 30.0
READOUT_TIME = 2.0
SLOT_TIME = EXPOSURE_TIME + READOUT_TIME
MAX_SLEW_SPEED_ALT = 3.5
MAX_SLEW_SPEED_AZ = 7.0
SLEW_SETTLE_TIME = 2.0
PHOTON_FLUX_MAG20_ZENITH = 100.0
RUBIN_EFFECTIVE_AREA = np.pi * (6.4 / 2) ** 2
RUBIN_QUANTUM_EFFICIENCY = 0.8
MAX_SIGMA_MAG = 0.3
MAX_FLAG_VALUE = 0
PATCHINESS_SAMPLE_N = 10
PATCHINESS_LOW_FRAC = 0.10
PATCHINESS_HIGH_FRAC = 0.90
PATCHINESS_VAR_THRESH = 0.05
URL_COL = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"

# ======================
# UTILITIES (identical to pipeline)
# ======================
def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url

def fetch_sys_map(url):
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    bio = io.BytesIO(data)
    with h5py.File(bio, "r") as f:
        clouds = np.array(f["clouds"]).ravel()
        sigma = np.array(f["sigma"]).ravel()
        flags = np.array(f["flags"]).ravel()
        mask_visible = np.array(f["mask_visible"]).ravel()
        nobs = np.array(f["nobs"]).ravel()
    bad = (~mask_visible) | (nobs == 0) | (flags > MAX_FLAG_VALUE) | (sigma > MAX_SIGMA_MAG) | (~np.isfinite(clouds))
    clouds[bad] = np.nan
    sigma[bad] = np.nan
    return clouds, sigma

def _make_location():
    return EarthLocation(lat=RUBIN_LAT*u.deg, lon=RUBIN_LON*u.deg, height=RUBIN_HEIGHT_M*u.m)

def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc

def radec_to_altaz_xy(ra_deg, dec_deg, obstime):
    t = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az = aa.az.deg % 360.0
    if alt < MIN_ALT_DEG:
        return None, None, alt, az
    alt_r = np.radians(alt)
    az_r = np.radians(az)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return x, y, alt, az

def altaz_to_xy(alt_deg, az_deg):
    if alt_deg < MIN_ALT_DEG:
        return None, None
    alt_r = np.radians(alt_deg)
    az_r = np.radians(az_deg)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return x, y

def xy_to_altaz(x_km, y_km):
    r = np.sqrt(x_km**2 + y_km**2)
    alt = np.degrees(np.arctan2(R_PROJECTION, r))
    az = np.degrees(np.arctan2(-x_km, y_km)) % 360.0
    return alt, az

def get_moon_altaz(obstime):
    t = _ensure_time(obstime)
    loc = _make_location()
    moon = get_body("moon", t, loc)
    aa = moon.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)

def angular_separation(alt1, az1, alt2, az2):
    a1, a2 = np.radians(alt1), np.radians(alt2)
    z1, z2 = np.radians(az1), np.radians(az2)
    cos_sep = np.sin(a1)*np.sin(a2) + np.cos(a1)*np.cos(a2)*np.cos(z1-z2)
    cos_sep = np.clip(cos_sep, -1.0, 1.0)
    return np.degrees(np.arccos(cos_sep))

def is_moon_safe(alt, az, moon_alt, moon_az):
    if alt < MIN_ALT_DEG:
        return False
    return angular_separation(alt, az, moon_alt, moon_az) >= MOON_EXCLUSION_DEG

def healpix_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    npix = hp.nside2npix(NSIDE_EXPECTED)
    pix = np.arange(npix)
    theta, phi = hp.pix2ang(NSIDE_EXPECTED, pix, nest=NEST)
    ra = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)
    t = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az = aa.az.deg % 360.0
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)

    alt_r = np.radians(alt)
    az_r = np.radians(az)
    above = alt > MIN_ALT_DEG
    scale = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf = -np.cos(alt_r) * np.sin(az_r) * scale
    yf =  np.cos(alt_r) * np.cos(az_r) * scale
    vf = np.where(above, vals, np.nan)

    r = np.sqrt(xf**2 + yf**2)
    c = r <= 15000.0
    ok = ~np.isnan(vf[c])

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)
    Hs, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges], weights=vf[c][ok])
    Hc, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15000] = np.nan
    return H, Xg, Yg, x_edges, y_edges

def calculate_slew_time(alt1, az1, alt2, az2):
    da = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_SPEED_ALT, daz / MAX_SLEW_SPEED_AZ) + SLEW_SETTLE_TIME

def compute_photon_collection(ext_mag):
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    rate = PHOTON_FLUX_MAG20_ZENITH * RUBIN_EFFECTIVE_AREA * RUBIN_QUANTUM_EFFICIENCY * flux
    return rate * EXPOSURE_TIME

def photons_to_magnitude(photons):
    if np.isnan(photons) or photons <= 0:
        return np.nan
    snr = photons / (5 * np.sqrt(photons))
    return 20.0 - 2.5 * np.log10(snr) if snr > 0 else np.nan

def get_value_at_position(grid, x_km, y_km):
    xi = int(round((x_km / BIN_SIZE_KM) + 15))
    yi = int(round((y_km / BIN_SIZE_KM) + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan

def find_absolute_minimum(grid, moon_alt=None, moon_az=None):
    work = grid.copy()
    if moon_alt is not None and moon_az is not None:
        ny, nx = work.shape
        for yi in range(ny):
            for xi in range(nx):
                if np.isnan(work[yi, xi]):
                    continue
                x_km = (xi - 15) * BIN_SIZE_KM
                y_km = (yi - 15) * BIN_SIZE_KM
                alt, az = xy_to_altaz(x_km, y_km)
                if not is_moon_safe(alt, az, moon_alt, moon_az):
                    work[yi, xi] = np.nan
    if not np.any(~np.isnan(work)):
        work = grid.copy()
    if not np.any(~np.isnan(work)):
        return 0, 0, np.nan
    idx = np.nanargmin(work)
    yi, xi = np.unravel_index(idx, work.shape)
    x = (xi - 15) * BIN_SIZE_KM
    y = (yi - 15) * BIN_SIZE_KM
    return x, y, grid[yi, xi]

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)
    g1s = gaussian_filter(g1, sigma=sigma)
    g2s = gaussian_filter(g2, sigma=sigma)
    m1 = ~np.isnan(grid1) & (grid1 != 0)
    m2 = ~np.isnan(grid2) & (grid2 != 0)
    best_corr, best_dx, best_dy = -np.inf, 0, 0
    for dy in range(-search_range, search_range+1):
        for dx in range(-search_range, search_range+1):
            sh = ndimage.shift(g2s, (dy, dx), order=1, mode='constant', cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode='constant', cval=0) > 0.5
            valid = m1 & sm
            if np.sum(valid) < 100:
                continue
            v1, v2 = g1s[valid], sh[valid]
            if np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0,1]
                if corr > best_corr:
                    best_corr, best_dx, best_dy = corr, dx, dy
    return best_dx, best_dy, best_corr

def score_candidate(x_km, y_km, ext_val, prev_alt, prev_az, moon_alt, moon_az):
    if np.isnan(ext_val):
        return np.inf
    alt, az = xy_to_altaz(x_km, y_km)
    if alt < MIN_ALT_DEG:
        return np.inf
    if not is_moon_safe(alt, az, moon_alt, moon_az):
        return np.inf
    cloud_norm = np.clip(ext_val / MAX_EXTINCTION_NORM, 0.0, 1.0)
    if prev_alt is not None:
        slew = calculate_slew_time(prev_alt, prev_az, alt, az)
    else:
        slew = 0.0
    slew_norm = np.clip(slew / MAX_SLEW_NORM, 0.0, 1.0)
    zenith_norm = (90.0 - alt) / 90.0
    return W_CLOUD * cloud_norm + W_SLEW * slew_norm + W_ZENITH * zenith_norm

def predict_future_position(cx, cy, dx_pix, dy_pix, current_grid, prev_alt, prev_az, moon_alt, moon_az):
    nx = cx - dx_pix * BIN_SIZE_KM
    ny = cy - dy_pix * BIN_SIZE_KM
    r = np.sqrt(nx**2 + ny**2)
    if r > 14000:
        nx *= 14000 / r
        ny *= 14000 / r
    candidates = [(nx, ny)]
    for dy in [-1,0,1]:
        for dx in [-1,0,1]:
            if dx==0 and dy==0: continue
            candidates.append((nx + dx*BIN_SIZE_KM, ny + dy*BIN_SIZE_KM))
    best_score = np.inf
    best_x, best_y = nx, ny
    for cx_, cy_ in candidates:
        ext = get_value_at_position(current_grid, cx_, cy_)
        s = score_candidate(cx_, cy_, ext, prev_alt, prev_az, moon_alt, moon_az)
        if s < best_score:
            best_score = s
            best_x, best_y = cx_, cy_
    if best_score == np.inf:
        best_x, best_y, _ = find_absolute_minimum(current_grid, moon_alt, moon_az)
    return best_x, best_y

# ======================
# DATA LOADING
# ======================
def load_night_sys_frames(csv_file, night_key):
    all_sys = load_all_sys_frames(csv_file)
    df_night = all_sys[all_sys["night_key"] == night_key].copy().reset_index(drop=True)
    return df_night

def load_all_sys_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', '').str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains('.hdf5', case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains('cloud_sys', case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    shifted = df[TIME_COL] - pd.Timedelta(hours=12)
    df["night_key"] = shifted.dt.date
    return df

def load_scheduler_observations(db_file):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    table_name = next((n for n in ['observations','SummaryAllProps','Summary','obs'] if n in tables), tables[0])
    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    night_col = next((c for c in obs_df.columns if 'night' in c.lower()), None)
    if night_col is None:
        raise RuntimeError("No night column")
    obs_df['night'] = obs_df[night_col].astype(int)
    for req in ['observationStartMJD','fieldRA','fieldDec']:
        for col in obs_df.columns:
            if req.lower() in col.lower():
                obs_df[req] = obs_df[col]
                break
    # pick the night with most observations (typical scheduler night)
    best_night = obs_df.groupby('night').size().idxmax()
    night_obs = obs_df[obs_df['night'] == best_night].copy()
    night_obs = night_obs.sort_values('observationStartMJD').reset_index(drop=True)
    print(f"  Scheduler night {best_night} with {len(night_obs)} obs")
    return night_obs

def match_scheduler_to_frames(scheduler_obs, frame_times):
    n = len(frame_times)
    fracs = np.linspace(0, 1, n)
    mjd = scheduler_obs['observationStartMJD'].values
    mjd_range = mjd.max() - mjd.min()
    s_fracs = (mjd - mjd.min()) / mjd_range if mjd_range > 0 else np.zeros(len(mjd))
    matched = [scheduler_obs.iloc[np.argmin(np.abs(s_fracs - f))] for f in fracs]
    return matched

def compute_all_positions(all_grids, all_metas):
    t0 = all_metas[0]["time"]
    m_alt0, m_az0 = get_moon_altaz(t0)
    x0, y0, _ = find_absolute_minimum(all_grids[0], m_alt0, m_az0)
    abs_pos = [(x0, y0)]
    pred_pos = [(x0, y0)]
    motion_v = [(0,0)]
    moon_pos = [(m_alt0, m_az0)]
    xp, yp = x0, y0
    prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    HL = 3
    for i in range(1, len(all_grids)):
        m_alt, m_az = get_moon_altaz(all_metas[i]["time"])
        moon_pos.append((m_alt, m_az))
        xa, ya, _ = find_absolute_minimum(all_grids[i], m_alt, m_az)
        abs_pos.append((xa, ya))
        if i >= HL:
            dxs, dys = [], []
            for j in range(1, HL+1):
                dx, dy, conf = compute_motion_with_correlation(all_grids[i-j], all_grids[i-j+1])
                if conf > 0.5:
                    dxs.append(dx); dys.append(dy)
            dx_avg = np.mean(dxs) if dxs else 0.0
            dy_avg = np.mean(dys) if dys else 0.0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(all_grids[i-1], all_grids[i])
        motion_v.append((dx_avg, dy_avg))
        xp, yp = predict_future_position(xp, yp, dx_avg, dy_avg, all_grids[i],
                                         prev_alt_p, prev_az_p, m_alt, m_az)
        pred_pos.append((xp, yp))
        prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    return abs_pos, pred_pos, motion_v, moon_pos

def compute_metrics(all_grids, all_sigma, all_metas, abs_pos, pred_pos, moon_pos, matched_sched):
    metrics = {s: {"photons":[], "sigma":[], "zenith_angles":[], "extinctions":[],
                   "slew_times":[], "expose_times":[], "frame_indices":[],
                   "alt_az":[], "xy_km":[]}
               for s in ("absolute","motion","scheduler")}
    prev_alt = {s:None for s in metrics}
    prev_az  = {s:None for s in metrics}
    for i in range(len(all_grids)):
        grid = all_grids[i]
        sgrid = all_sigma[i]
        m_alt, m_az = moon_pos[i]
        # absolute
        xa, ya = abs_pos[i]
        alt_a, az_a = xy_to_altaz(xa, ya)
        if alt_a >= MIN_ALT_DEG and is_moon_safe(alt_a, az_a, m_alt, m_az):
            ext = get_value_at_position(grid, xa, ya)
            sig = get_value_at_position(sgrid, xa, ya)
            za = 90.0 - alt_a
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["absolute"], prev_az["absolute"], alt_a, az_a) if prev_alt["absolute"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["absolute"]["photons"].append(collected)
                metrics["absolute"]["sigma"].append(sig)
                metrics["absolute"]["zenith_angles"].append(za)
                metrics["absolute"]["extinctions"].append(ext)
                metrics["absolute"]["slew_times"].append(slew)
                metrics["absolute"]["expose_times"].append(expose_t)
                metrics["absolute"]["frame_indices"].append(i)
                metrics["absolute"]["alt_az"].append((alt_a, az_a))
                metrics["absolute"]["xy_km"].append((xa, ya))
                prev_alt["absolute"], prev_az["absolute"] = alt_a, az_a
        # motion
        xp, yp = pred_pos[i]
        alt_p, az_p = xy_to_altaz(xp, yp)
        if alt_p >= MIN_ALT_DEG and is_moon_safe(alt_p, az_p, m_alt, m_az):
            ext = get_value_at_position(grid, xp, yp)
            sig = get_value_at_position(sgrid, xp, yp)
            za = 90.0 - alt_p
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["motion"], prev_az["motion"], alt_p, az_p) if prev_alt["motion"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["motion"]["photons"].append(collected)
                metrics["motion"]["sigma"].append(sig)
                metrics["motion"]["zenith_angles"].append(za)
                metrics["motion"]["extinctions"].append(ext)
                metrics["motion"]["slew_times"].append(slew)
                metrics["motion"]["expose_times"].append(expose_t)
                metrics["motion"]["frame_indices"].append(i)
                metrics["motion"]["alt_az"].append((alt_p, az_p))
                metrics["motion"]["xy_km"].append((xp, yp))
                prev_alt["motion"], prev_az["motion"] = alt_p, az_p
        # scheduler
        obs = matched_sched[i]
        xs, ys, alt_s, az_s = radec_to_altaz_xy(obs['fieldRA'], obs['fieldDec'], all_metas[i]["time"])
        if xs is not None:
            ext = get_value_at_position(grid, xs, ys)
            sig = get_value_at_position(sgrid, xs, ys)
            za = 90.0 - alt_s
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["scheduler"], prev_az["scheduler"], alt_s, az_s) if prev_alt["scheduler"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["scheduler"]["photons"].append(collected)
                metrics["scheduler"]["sigma"].append(sig)
                metrics["scheduler"]["zenith_angles"].append(za)
                metrics["scheduler"]["extinctions"].append(ext)
                metrics["scheduler"]["slew_times"].append(slew)
                metrics["scheduler"]["expose_times"].append(expose_t)
                metrics["scheduler"]["frame_indices"].append(i)
                metrics["scheduler"]["alt_az"].append((alt_s, az_s))
                metrics["scheduler"]["xy_km"].append((xs, ys))
                prev_alt["scheduler"], prev_az["scheduler"] = alt_s, az_s
    return metrics

# ======================
# METRICS PLOT (ONLY FIRST 3 ROWS)
# ======================
def create_metrics_plot(metrics, output_png):
    strategies = ("absolute", "motion", "scheduler")
    colors = ("green", "blue", "red")
    labels = ("Absolute Min", "Motion Tracking", "Scheduler")

    # Figure with 3 rows, 3 columns (row0: histograms, row1: timeline, row2: bar charts)
    fig = plt.figure(figsize=(18, 10))   # shorter than original because bottom row removed
    gs = fig.add_gridspec(3, 3, hspace=0.40, wspace=0.32)

    # Row 0: Zenith angle histogram
    ax = fig.add_subplot(gs[0,0])
    for s,c,l in zip(strategies, colors, labels):
        if metrics[s]["zenith_angles"]:
            ax.hist(metrics[s]["zenith_angles"], bins=20, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Zenith Angle (°)"); ax.set_ylabel("Count")
    ax.set_title("Zenith Angle Distribution", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    # Row 0: Slew time histogram
    ax = fig.add_subplot(gs[0,1])
    for s,c,l in zip(strategies, colors, labels):
        slew = metrics[s]["slew_times"][1:]  # exclude first zero
        if slew:
            ax.hist(slew, bins=30, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Slew Time (s)"); ax.set_ylabel("Count")
    ax.set_title("Slew Time Distribution", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    # Row 0: Sigma histogram
    ax = fig.add_subplot(gs[0,2])
    for s,c,l in zip(strategies, colors, labels):
        sigs = [x for x in metrics[s]["sigma"] if not np.isnan(x)]
        if sigs:
            ax.hist(sigs, bins=30, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.axvline(MAX_SIGMA_MAG, color='k', linestyle='--', lw=1.5, label=f"Mask threshold ({MAX_SIGMA_MAG})")
    ax.set_xlabel("Pixel σ uncertainty (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Uncertainty", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    # Row 1: Photon timeline (full width)
    ax = fig.add_subplot(gs[1,:])
    for s,c,l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        ph = metrics[s]["photons"]
        if fi:
            ax.plot(fi, ph, color=c, alpha=0.7, linewidth=1.5, label=l)
    ax.set_xlabel("Frame Number (Time →)"); ax.set_ylabel("Slew-gated photons per slot")
    ax.set_title("Photon Collection Over the Night", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

    # Row 2: Total photons
    ax = fig.add_subplot(gs[2,0])
    x_pos = np.arange(len(strategies))
    width = 0.6
    totals = [np.sum(metrics[s]["photons"]) if metrics[s]["photons"] else 0 for s in strategies]
    bars = ax.bar(x_pos, totals, width, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    baseline = totals[0] if totals[0]>0 else 1
    for i, t in enumerate(totals):
        ax.text(i, t, f"{((t-baseline)/baseline)*100:+.1f}%", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_xticks(x_pos); ax.set_xticklabels(labels)
    ax.set_ylabel("Total Photons"); ax.set_title("Total Night Photon Collection", weight="bold")
    ax.grid(alpha=0.3, axis='y')
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

    # Row 2: Median depth
    ax = fig.add_subplot(gs[2,1])
    depths = []
    for s in strategies:
        mags = [photons_to_magnitude(p) for p in metrics[s]["photons"]]
        mags = [m for m in mags if not np.isnan(m)]
        depths.append(np.median(mags) if mags else 0)
    bars = ax.bar(labels, depths, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for bar, val in zip(bars, depths):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.2f}", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_ylabel("Median 5σ Depth (mag)"); ax.set_title("Survey Depth (slew-scaled)", weight="bold")
    ax.grid(alpha=0.3, axis='y'); ax.invert_yaxis()

    # Row 2: Efficiency
    ax = fig.add_subplot(gs[2,2])
    effs = []
    for s in strategies:
        et = np.array(metrics[s]["expose_times"])
        slots = len(et) * SLOT_TIME
        effs.append(np.sum(et) / slots * 100 if slots>0 else 0)
    bars = ax.bar(labels, effs, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for bar, val in zip(bars, effs):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.1f}%", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_ylabel("Shutter-open efficiency (%)"); ax.set_title("Observation Efficiency", weight="bold")
    ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0,105)

    plt.suptitle("Rubin Observatory Pointing Strategy Comparison\n(σ-masked, slew-gated, moon-avoided, zenith-weighted)",
                 fontsize=16, weight="bold", y=0.98)
    plt.savefig(output_png, dpi=150, bbox_inches="tight")
    print(f"  Metrics plot saved: {output_png}")
    plt.close()

# ======================
# VIDEO (optional, but kept for completeness)
# ======================
def create_video(all_grids, all_sigma, all_metas,
                 abs_positions, pred_positions, motion_vectors, moon_positions,
                 matched_scheduler, x_edges, y_edges,
                 output_file, fps=10):
    print(f"  Creating video {output_file} ...")
    sched_xy = []
    for i, obs in enumerate(matched_scheduler):
        x, y, alt, az = radec_to_altaz_xy(obs['fieldRA'], obs['fieldDec'], all_metas[i]["time"])
        sched_xy.append((x, y, alt, az))

    fig, ax_map = plt.subplots(1, 1, figsize=(12, 10))

    def update(fi):
        ax_map.clear()
        grid = all_grids[fi]
        m_alt, m_az = moon_positions[fi]
        ax_map.pcolormesh(x_edges, y_edges, grid, cmap="viridis", vmin=-0.2, vmax=2.0, shading='flat', alpha=0.85)
        th = np.linspace(0, 2*np.pi, 300)
        ax_map.plot(15000*np.cos(th), 15000*np.sin(th), 'k-', lw=2, alpha=0.5)
        ax_map.plot(0, 0, 'r+', ms=15, mew=3, label="Zenith", zorder=5)
        if m_alt > 0:
            moon_x, moon_y = altaz_to_xy(m_alt, m_az)
            if moon_x is not None:
                excl_r = MOON_EXCLUSION_DEG / 90.0 * R_PROJECTION
                ax_map.plot(moon_x + excl_r*np.cos(th), moon_y + excl_r*np.sin(th), 'y--', lw=1.5, alpha=0.7)
                ax_map.plot(moon_x, moon_y, 'y*', ms=14, zorder=6)
        # absolute
        xa, ya = abs_positions[fi]
        va = get_value_at_position(grid, xa, ya)
        ax_map.plot(xa, ya, 'go', ms=20, mew=2.5, mec='white', label=f"Abs Min  ext={va:.2f}", zorder=10)
        # motion
        xp, yp = pred_positions[fi]
        vp = get_value_at_position(grid, xp, yp)
        ax_map.plot(xp, yp, 'bo', ms=20, mew=2.5, mec='white', label=f"Motion   ext={vp:.2f}", zorder=10)
        # arrow
        if fi > 0:
            dx_pix, dy_pix = motion_vectors[fi]
            dxk = dx_pix * BIN_SIZE_KM
            dyk = dy_pix * BIN_SIZE_KM
            mag = np.sqrt(dxk**2 + dyk**2)
            if mag > 300:
                arrow = FancyArrowPatch((xp, yp), (xp+dxk, yp+dyk), arrowstyle='->', mutation_scale=28, lw=2.5, color='cyan', alpha=0.9, zorder=15)
                ax_map.add_patch(arrow)
        # scheduler
        xs, ys, _, _ = sched_xy[fi]
        if xs is not None:
            vs = get_value_at_position(grid, xs, ys)
            ax_map.plot(xs, ys, 'ro', ms=20, mew=2.5, mec='white', label=f"Scheduler ext={vs:.2f}", zorder=10)
        ax_map.set_xlabel("X (km)"); ax_map.set_ylabel("Y (km)")
        ax_map.set_aspect("equal"); ax_map.grid(alpha=0.2)
        ax_map.set_xlim(-16000, 16000); ax_map.set_ylim(-16000, 16000)
        ts_str = all_metas[fi]["time"].strftime("%Y-%m-%d %H:%M:%S UTC")
        ax_map.set_title(f"Cloud Coverage — {ts_str}\nFrame {fi+1}/{len(all_grids)}", fontsize=12, weight="bold")
        ax_map.legend(loc="upper right", fontsize=8.5, framealpha=0.9)
        return []

    anim = FuncAnimation(fig, update, frames=len(all_grids), interval=1000/fps, blit=False, repeat=False)
    writer = writers['ffmpeg'](fps=fps, bitrate=5000, codec='libx264', extra_args=['-pix_fmt', 'yuv420p'])
    anim.save(output_file, writer=writer)
    print(f"  Video saved: {output_file}")
    plt.close()

# ======================
# MAIN LOOP OVER NIGHTS (REVERSE ORDER)
# ======================
def main():
    csv_file = "feb5_data.csv"
    db_file = "baseline_v5.1.0_10yrs.db"
    output_dir = "."
    nights = [
        "2025-06-27", "2025-06-30", "2025-07-01", "2025-07-14",
        "2025-07-15", "2025-07-20", "2025-07-21", "2025-07-23",
        "2025-07-29", "2025-07-30", "2025-08-09", "2025-08-12"
    ]
    nights_rev = list(reversed(nights))
    print("Processing nights in reverse order:")
    for n in nights_rev:
        print(f"  {n}")

    # Load scheduler once (same for all nights)
    scheduler_obs = load_scheduler_observations(db_file)

    for night_key in nights_rev:
        tag = night_key.replace("-", "")
        print(f"\n{'█'*70}\n  NIGHT: {night_key}\n{'█'*70}")
        df_night = load_night_sys_frames(csv_file, night_key)
        if len(df_night) < 5:
            print("  Too few frames, skipping.")
            continue

        # Quick patchiness probe (using first 10 frames)
        print("  [Patchiness probe]")
        patchy, reason = quick_patchiness_check(df_night)
        if not patchy:
            print(f"  Skipping night (not patchy): {reason}")
            continue
        print("  Patchy clouds confirmed, loading all frames...")

        # Load all grids, sigma, metas
        all_grids = []
        all_sigma = []
        all_metas = []
        x_edges = y_edges = None
        for _, row in df_night.iterrows():
            url = transform_url(row[URL_COL])
            try:
                clouds, sigma = fetch_sys_map(url)
                obstime = row[TIME_COL].to_pydatetime()
                grid, Xg, Yg, xe, ye = healpix_to_grid(clouds, obstime)
                sig_grid, _, _, _, _ = healpix_to_grid(sigma, obstime)
                all_grids.append(grid)
                all_sigma.append(sig_grid)
                all_metas.append({"time": obstime})
                if x_edges is None:
                    x_edges, y_edges = xe, ye
                if len(all_grids) % 100 == 0:
                    print(f"    Loaded {len(all_grids)} frames ...")
            except Exception as e:
                print(f"    Frame load failed: {e}")
        print(f"  Loaded {len(all_grids)} frames total.")
        if len(all_grids) < 2:
            continue

        # Match scheduler
        matched_sched = match_scheduler_to_frames(scheduler_obs, [m["time"] for m in all_metas])

        # Motion tracking
        print("  Computing cloud motion tracking...")
        abs_pos, pred_pos, motion_v, moon_pos = compute_all_positions(all_grids, all_metas)

        # Metrics
        print("  Computing metrics...")
        metrics = compute_metrics(all_grids, all_sigma, all_metas,
                                  abs_pos, pred_pos, moon_pos, matched_sched)

        # Print quick summary
        for s in ("absolute","motion","scheduler"):
            ph = np.array(metrics[s]["photons"])
            if len(ph):
                depth = np.median([photons_to_magnitude(p) for p in ph if not np.isnan(p)])
                print(f"    {s:10s}: {len(ph)} slots, total photons={np.sum(ph):.3e}, median depth={depth:.2f} mag")

        # Metrics plot (no bottom row)
        plot_file = os.path.join(output_dir, f"metrics_{tag}.png")
        create_metrics_plot(metrics, plot_file)

        # Video (optional, but included for completeness)
        video_file = os.path.join(output_dir, f"cloud_tracking_{tag}.mp4")
        create_video(all_grids, all_sigma, all_metas,
                     abs_pos, pred_pos, motion_v, moon_pos,
                     matched_sched, x_edges, y_edges,
                     video_file, fps=10)

        # Pointing tensor (optional)
        tensor = build_pointing_tensor(metrics, all_metas)
        tensor_file = os.path.join(output_dir, f"pointing_tensor_{tag}.npz")
        save_pointing_tensor(tensor, tensor_file, night_key)

        print(f"  ✓ Night {night_key} done.\n")

def quick_patchiness_check(df_night, n_probe=10):
    if len(df_night) == 0:
        return False, "No frames"
    idx = np.linspace(0, len(df_night)-1, min(n_probe, len(df_night)), dtype=int)
    grids = []
    for i in idx:
        row = df_night.iloc[i]
        url = transform_url(row[URL_COL])
        try:
            clouds, _ = fetch_sys_map(url)
            obstime = row[TIME_COL].to_pydatetime()
            grid, _, _, _, _ = healpix_to_grid(clouds, obstime)
            grids.append(grid)
        except Exception:
            continue
    if len(grids) < 2:
        return False, "Too few probe frames"
    # compute mean valid fraction and spatial variance
    fracs = []
    vars_ = []
    for g in grids:
        valid = ~np.isnan(g)
        frac = np.sum(valid) / g.size
        var = np.nanvar(g) if np.sum(valid) > 10 else 0.0
        fracs.append(frac)
        vars_.append(var)
    mean_frac = np.mean(fracs)
    mean_var = np.mean(vars_)
    if mean_frac < PATCHINESS_LOW_FRAC:
        return False, f"mostly clear (frac={mean_frac:.2f})"
    if mean_frac > PATCHINESS_HIGH_FRAC:
        return False, f"mostly cloudy (frac={mean_frac:.2f})"
    if mean_var < PATCHINESS_VAR_THRESH:
        return False, f"uniform cover (var={mean_var:.4f})"
    return True, f"patchy (frac={mean_frac:.2f}, var={mean_var:.4f})"

def build_pointing_tensor(metrics, all_metas):
    strategy_ids = {"absolute":0, "motion":1, "scheduler":2}
    rows = []
    for strategy, sid in strategy_ids.items():
        for k in range(len(metrics[strategy]["frame_indices"])):
            fi = metrics[strategy]["frame_indices"][k]
            x_km, y_km = metrics[strategy]["xy_km"][k]
            alt, az = xy_to_altaz(x_km, y_km)
            t_unix = all_metas[fi]["time"].timestamp()
            ext = metrics[strategy]["extinctions"][k]
            rows.append([float(fi), t_unix, alt, az, x_km, y_km, float(ext), float(sid)])
    if not rows:
        return np.zeros((0,8), dtype=np.float32)
    return np.array(rows, dtype=np.float32)

def save_pointing_tensor(tensor, output_path, night_key):
    np.savez_compressed(output_path, pointing_tensor=tensor,
                        night_key=np.array([str(night_key)]),
                        columns=np.array(["frame_idx","unix_time","alt_deg","az_deg",
                                          "x_km","y_km","extinction","strategy_id"]))
    print(f"  Pointing tensor saved: {output_path} ({tensor.shape[0]} rows)")

if __name__ == "__main__":
    main()

Processing nights in reverse order:
  2025-08-12
  2025-08-09
  2025-07-30
  2025-07-29
  2025-07-23
  2025-07-21
  2025-07-20
  2025-07-15
  2025-07-14
  2025-07-01
  2025-06-30
  2025-06-27
  Scheduler night 572 with 1139 obs

██████████████████████████████████████████████████████████████████████
  NIGHT: 2025-08-12
██████████████████████████████████████████████████████████████████████
  Too few frames, skipping.

██████████████████████████████████████████████████████████████████████
  NIGHT: 2025-08-09
██████████████████████████████████████████████████████████████████████
  Too few frames, skipping.

██████████████████████████████████████████████████████████████████████
  NIGHT: 2025-07-30
██████████████████████████████████████████████████████████████████████
  Too few frames, skipping.

██████████████████████████████████████████████████████████████████████
  NIGHT: 2025-07-29
██████████████████████████████████████████████████████████████████████
  Too few frames, skipping.

███████

In [18]:
#!/usr/bin/env python3
"""
Run full DREAM pipeline for multiple nights (starting from frame 210 for each night).
Produce standard metrics plots (histograms + timelines) with publication-ready styling,
without any atmospheric correction or extra bottom row.
Save plots as PDF/PNG and save metrics data for each night.
"""

import io
import os
import sqlite3
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.ndimage import gaussian_filter
from scipy import ndimage, stats
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from lsst.summit.utils import ConsDbClient

import warnings
warnings.filterwarnings("ignore")

# ======================
# PUBLICATION-READY STYLE
# ======================
mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "stix",
    "font.family": "STIXGeneral",
    "axes.labelsize": 22,
    "axes.titlesize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 28,
    "lines.linewidth": 2.0,
    "axes.linewidth": 1.3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
})

# ======================
# CONSTANTS (unchanged)
# ======================
NSIDE_EXPECTED = 32
NEST = True
UNSEEN = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan
RUBIN_LAT = -30.244639
RUBIN_LON = -70.749417
RUBIN_HEIGHT_M = 2663.0
BIN_SIZE_KM = 1000
R_PROJECTION = 10000.0
MIN_ALT_DEG = 15.0
MOON_EXCLUSION_DEG = 30.0
W_CLOUD = 0.50
W_SLEW = 0.25
W_ZENITH = 0.25
MAX_EXTINCTION_NORM = 2.0
MAX_SLEW_NORM = 60.0
EXPOSURE_TIME = 30.0
READOUT_TIME = 2.0
SLOT_TIME = EXPOSURE_TIME + READOUT_TIME
MAX_SLEW_SPEED_ALT = 3.5
MAX_SLEW_SPEED_AZ = 7.0
SLEW_SETTLE_TIME = 2.0
PHOTON_FLUX_MAG20_ZENITH = 100.0
RUBIN_EFFECTIVE_AREA = np.pi * (6.4 / 2) ** 2
RUBIN_QUANTUM_EFFICIENCY = 0.8
MAX_SIGMA_MAG = 0.3
MAX_FLAG_VALUE = 0
PATCHINESS_SAMPLE_N = 10
PATCHINESS_LOW_FRAC = 0.10
PATCHINESS_HIGH_FRAC = 0.90
PATCHINESS_VAR_THRESH = 0.05
ALPHA_SEEING = 0.6
BETA_SEEING = 0.2
FWHM_CONSTANT = 2 * np.sqrt(2 * np.log(2))
EFFECTIVE_WAVELENGTHS = {'u':366.3, 'g':481.0, 'r':622.2, 'i':754.5, 'z':869.1, 'y':971.0}
URL_COL = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"

# ======================
# UTILITY FUNCTIONS (unchanged)
# ======================
def transform_url(url):
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url

def fetch_sys_map(url):
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    bio = io.BytesIO(data)
    with h5py.File(bio, "r") as f:
        clouds = np.array(f["clouds"]).ravel()
        sigma = np.array(f["sigma"]).ravel()
        flags = np.array(f["flags"]).ravel()
        mask_visible = np.array(f["mask_visible"]).ravel()
        nobs = np.array(f["nobs"]).ravel()
    bad = (~mask_visible) | (nobs == 0) | (flags > MAX_FLAG_VALUE) | (sigma > MAX_SIGMA_MAG) | (~np.isfinite(clouds))
    clouds[bad] = np.nan
    sigma[bad] = np.nan
    return clouds, sigma

def _make_location():
    return EarthLocation(lat=RUBIN_LAT*u.deg, lon=RUBIN_LON*u.deg, height=RUBIN_HEIGHT_M*u.m)

def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc

def radec_to_altaz_xy(ra_deg, dec_deg, obstime):
    t = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az = aa.az.deg % 360.0
    if alt < MIN_ALT_DEG:
        return None, None, alt, az
    alt_r = np.radians(alt)
    az_r = np.radians(az)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return x, y, alt, az

def altaz_to_xy(alt_deg, az_deg):
    if alt_deg < MIN_ALT_DEG:
        return None, None
    alt_r = np.radians(alt_deg)
    az_r = np.radians(az_deg)
    scale = R_PROJECTION / np.sin(alt_r)
    x = -np.cos(alt_r) * np.sin(az_r) * scale
    y =  np.cos(alt_r) * np.cos(az_r) * scale
    return x, y

def xy_to_altaz(x_km, y_km):
    r = np.sqrt(x_km**2 + y_km**2)
    alt = np.degrees(np.arctan2(R_PROJECTION, r))
    az = np.degrees(np.arctan2(-x_km, y_km)) % 360.0
    return alt, az

def get_moon_altaz(obstime):
    t = _ensure_time(obstime)
    loc = _make_location()
    moon = get_body("moon", t, loc)
    aa = moon.transform_to(AltAz(obstime=t, location=loc))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)

def angular_separation(alt1, az1, alt2, az2):
    a1, a2 = np.radians(alt1), np.radians(alt2)
    z1, z2 = np.radians(az1), np.radians(az2)
    cos_sep = np.sin(a1)*np.sin(a2) + np.cos(a1)*np.cos(a2)*np.cos(z1-z2)
    cos_sep = np.clip(cos_sep, -1.0, 1.0)
    return np.degrees(np.arccos(cos_sep))

def is_moon_safe(alt, az, moon_alt, moon_az):
    if alt < MIN_ALT_DEG:
        return False
    return angular_separation(alt, az, moon_alt, moon_az) >= MOON_EXCLUSION_DEG

def healpix_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    npix = hp.nside2npix(NSIDE_EXPECTED)
    pix = np.arange(npix)
    theta, phi = hp.pix2ang(NSIDE_EXPECTED, pix, nest=NEST)
    ra = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)
    t = _ensure_time(obstime)
    loc = _make_location()
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa = sky.transform_to(AltAz(obstime=t, location=loc))
    alt = aa.alt.deg
    az = aa.az.deg % 360.0
    vals = np.asarray(mp, dtype=float)
    vals = np.where(vals == UNSEEN, np.nan, vals)

    alt_r = np.radians(alt)
    az_r = np.radians(az)
    above = alt > MIN_ALT_DEG
    scale = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf = -np.cos(alt_r) * np.sin(az_r) * scale
    yf =  np.cos(alt_r) * np.cos(az_r) * scale
    vf = np.where(above, vals, np.nan)

    r = np.sqrt(xf**2 + yf**2)
    c = r <= 15000.0
    ok = ~np.isnan(vf[c])

    x_edges = np.arange(-15000, 15001, bin_size)
    y_edges = np.arange(-15000, 15001, bin_size)
    Hs, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges], weights=vf[c][ok])
    Hc, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[x_edges, y_edges])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (x_edges[:-1] + x_edges[1:]) / 2
    yc = (y_edges[:-1] + y_edges[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15000] = np.nan
    return H, Xg, Yg, x_edges, y_edges

def calculate_slew_time(alt1, az1, alt2, az2):
    da = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180:
        daz = 360 - daz
    return max(da / MAX_SLEW_SPEED_ALT, daz / MAX_SLEW_SPEED_AZ) + SLEW_SETTLE_TIME

def compute_photon_collection(ext_mag):
    if np.isnan(ext_mag):
        return np.nan
    flux = 10 ** (-0.4 * ext_mag)
    rate = PHOTON_FLUX_MAG20_ZENITH * RUBIN_EFFECTIVE_AREA * RUBIN_QUANTUM_EFFICIENCY * flux
    return rate * EXPOSURE_TIME

def photons_to_magnitude(photons):
    if np.isnan(photons) or photons <= 0:
        return np.nan
    snr = photons / (5 * np.sqrt(photons))
    return 20.0 - 2.5 * np.log10(snr) if snr > 0 else np.nan

def get_value_at_position(grid, x_km, y_km):
    xi = int(round((x_km / BIN_SIZE_KM) + 15))
    yi = int(round((y_km / BIN_SIZE_KM) + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan

def find_absolute_minimum(grid, moon_alt=None, moon_az=None):
    work = grid.copy()
    if moon_alt is not None and moon_az is not None:
        ny, nx = work.shape
        for yi in range(ny):
            for xi in range(nx):
                if np.isnan(work[yi, xi]):
                    continue
                x_km = (xi - 15) * BIN_SIZE_KM
                y_km = (yi - 15) * BIN_SIZE_KM
                alt, az = xy_to_altaz(x_km, y_km)
                if not is_moon_safe(alt, az, moon_alt, moon_az):
                    work[yi, xi] = np.nan
    if not np.any(~np.isnan(work)):
        work = grid.copy()
    if not np.any(~np.isnan(work)):
        return 0, 0, np.nan
    idx = np.nanargmin(work)
    yi, xi = np.unravel_index(idx, work.shape)
    x = (xi - 15) * BIN_SIZE_KM
    y = (yi - 15) * BIN_SIZE_KM
    return x, y, grid[yi, xi]

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = np.nan_to_num(grid1, nan=0)
    g2 = np.nan_to_num(grid2, nan=0)
    g1s = gaussian_filter(g1, sigma=sigma)
    g2s = gaussian_filter(g2, sigma=sigma)
    m1 = ~np.isnan(grid1) & (grid1 != 0)
    m2 = ~np.isnan(grid2) & (grid2 != 0)
    best_corr, best_dx, best_dy = -np.inf, 0, 0
    for dy in range(-search_range, search_range+1):
        for dx in range(-search_range, search_range+1):
            sh = ndimage.shift(g2s, (dy, dx), order=1, mode='constant', cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0, mode='constant', cval=0) > 0.5
            valid = m1 & sm
            if np.sum(valid) < 100:
                continue
            v1, v2 = g1s[valid], sh[valid]
            if np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0,1]
                if corr > best_corr:
                    best_corr, best_dx, best_dy = corr, dx, dy
    return best_dx, best_dy, best_corr

def score_candidate(x_km, y_km, ext_val, prev_alt, prev_az, moon_alt, moon_az):
    if np.isnan(ext_val):
        return np.inf
    alt, az = xy_to_altaz(x_km, y_km)
    if alt < MIN_ALT_DEG:
        return np.inf
    if not is_moon_safe(alt, az, moon_alt, moon_az):
        return np.inf
    cloud_norm = np.clip(ext_val / MAX_EXTINCTION_NORM, 0.0, 1.0)
    if prev_alt is not None:
        slew = calculate_slew_time(prev_alt, prev_az, alt, az)
    else:
        slew = 0.0
    slew_norm = np.clip(slew / MAX_SLEW_NORM, 0.0, 1.0)
    zenith_norm = (90.0 - alt) / 90.0
    return W_CLOUD * cloud_norm + W_SLEW * slew_norm + W_ZENITH * zenith_norm

def predict_future_position(cx, cy, dx_pix, dy_pix, current_grid, prev_alt, prev_az, moon_alt, moon_az):
    nx = cx - dx_pix * BIN_SIZE_KM
    ny = cy - dy_pix * BIN_SIZE_KM
    r = np.sqrt(nx**2 + ny**2)
    if r > 14000:
        nx *= 14000 / r
        ny *= 14000 / r
    candidates = [(nx, ny)]
    for dy in [-1,0,1]:
        for dx in [-1,0,1]:
            if dx==0 and dy==0: continue
            candidates.append((nx + dx*BIN_SIZE_KM, ny + dy*BIN_SIZE_KM))
    best_score = np.inf
    best_x, best_y = nx, ny
    for cx_, cy_ in candidates:
        ext = get_value_at_position(current_grid, cx_, cy_)
        s = score_candidate(cx_, cy_, ext, prev_alt, prev_az, moon_alt, moon_az)
        if s < best_score:
            best_score = s
            best_x, best_y = cx_, cy_
    if best_score == np.inf:
        best_x, best_y, _ = find_absolute_minimum(current_grid, moon_alt, moon_az)
    return best_x, best_y

def load_night_sys_frames(csv_file, target_date, start_frame=0):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', '').str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains('.hdf5', case=False, na=False)].copy()
    df = df[df[URL_COL].str.contains('cloud_sys', case=False, na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    start = pd.Timestamp(f"{target_date} 00:00:00", tz="UTC")
    end   = pd.Timestamp(f"{target_date} 23:59:59", tz="UTC")
    df_night = df[(df[TIME_COL] >= start) & (df[TIME_COL] <= end)].copy()
    if len(df_night) <= start_frame:
        raise ValueError(f"Only {len(df_night)} frames, cannot start at index {start_frame}")
    df_night = df_night.iloc[start_frame:].reset_index(drop=True)
    print(f"Loaded {len(df_night)} frames for {target_date} (starting at original frame {start_frame})")
    return df_night

def load_scheduler_observations(db_file):
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    table_name = next((n for n in ['observations','SummaryAllProps','Summary','obs'] if n in tables), tables[0])
    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    night_col = next((c for c in obs_df.columns if 'night' in c.lower()), None)
    if night_col is None:
        raise RuntimeError("No night column")
    obs_df['night'] = obs_df[night_col].astype(int)
    for req in ['observationStartMJD','fieldRA','fieldDec']:
        for col in obs_df.columns:
            if req.lower() in col.lower():
                obs_df[req] = obs_df[col]
                break
    best_night = obs_df.groupby('night').size().idxmax()
    night_obs = obs_df[obs_df['night'] == best_night].copy()
    night_obs = night_obs.sort_values('observationStartMJD').reset_index(drop=True)
    print(f"Scheduler night {best_night} with {len(night_obs)} obs")
    return night_obs

def match_scheduler_to_frames(scheduler_obs, frame_times):
    n = len(frame_times)
    fracs = np.linspace(0, 1, n)
    mjd = scheduler_obs['observationStartMJD'].values
    mjd_range = mjd.max() - mjd.min()
    s_fracs = (mjd - mjd.min()) / mjd_range if mjd_range > 0 else np.zeros(len(mjd))
    matched = [scheduler_obs.iloc[np.argmin(np.abs(s_fracs - f))] for f in fracs]
    return matched

def compute_all_positions(all_grids, all_metas):
    t0 = all_metas[0]["time"]
    m_alt0, m_az0 = get_moon_altaz(t0)
    x0, y0, _ = find_absolute_minimum(all_grids[0], m_alt0, m_az0)
    abs_pos = [(x0, y0)]
    pred_pos = [(x0, y0)]
    motion_v = [(0,0)]
    moon_pos = [(m_alt0, m_az0)]
    xp, yp = x0, y0
    prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    HL = 3
    for i in range(1, len(all_grids)):
        m_alt, m_az = get_moon_altaz(all_metas[i]["time"])
        moon_pos.append((m_alt, m_az))
        xa, ya, _ = find_absolute_minimum(all_grids[i], m_alt, m_az)
        abs_pos.append((xa, ya))
        if i >= HL:
            dxs, dys = [], []
            for j in range(1, HL+1):
                dx, dy, conf = compute_motion_with_correlation(all_grids[i-j], all_grids[i-j+1])
                if conf > 0.5:
                    dxs.append(dx); dys.append(dy)
            dx_avg = np.mean(dxs) if dxs else 0.0
            dy_avg = np.mean(dys) if dys else 0.0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(all_grids[i-1], all_grids[i])
        motion_v.append((dx_avg, dy_avg))
        xp, yp = predict_future_position(xp, yp, dx_avg, dy_avg, all_grids[i],
                                         prev_alt_p, prev_az_p, m_alt, m_az)
        pred_pos.append((xp, yp))
        prev_alt_p, prev_az_p = xy_to_altaz(xp, yp)
    return abs_pos, pred_pos, motion_v, moon_pos

def compute_metrics(all_grids, all_sigma, all_metas, abs_pos, pred_pos, moon_pos, matched_sched):
    metrics = {s: {"photons":[], "sigma":[], "zenith_angles":[], "extinctions":[],
                   "slew_times":[], "expose_times":[], "frame_indices":[],
                   "alt_az":[], "xy_km":[]}
               for s in ("absolute","motion","scheduler")}
    prev_alt = {s:None for s in metrics}
    prev_az  = {s:None for s in metrics}
    for i in range(len(all_grids)):
        grid = all_grids[i]
        sgrid = all_sigma[i]
        m_alt, m_az = moon_pos[i]
        # absolute
        xa, ya = abs_pos[i]
        alt_a, az_a = xy_to_altaz(xa, ya)
        if alt_a >= MIN_ALT_DEG and is_moon_safe(alt_a, az_a, m_alt, m_az):
            ext = get_value_at_position(grid, xa, ya)
            sig = get_value_at_position(sgrid, xa, ya)
            za = 90.0 - alt_a
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["absolute"], prev_az["absolute"], alt_a, az_a) if prev_alt["absolute"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["absolute"]["photons"].append(collected)
                metrics["absolute"]["sigma"].append(sig)
                metrics["absolute"]["zenith_angles"].append(za)
                metrics["absolute"]["extinctions"].append(ext)
                metrics["absolute"]["slew_times"].append(slew)
                metrics["absolute"]["expose_times"].append(expose_t)
                metrics["absolute"]["frame_indices"].append(i)
                metrics["absolute"]["alt_az"].append((alt_a, az_a))
                metrics["absolute"]["xy_km"].append((xa, ya))
                prev_alt["absolute"], prev_az["absolute"] = alt_a, az_a
        # motion
        xp, yp = pred_pos[i]
        alt_p, az_p = xy_to_altaz(xp, yp)
        if alt_p >= MIN_ALT_DEG and is_moon_safe(alt_p, az_p, m_alt, m_az):
            ext = get_value_at_position(grid, xp, yp)
            sig = get_value_at_position(sgrid, xp, yp)
            za = 90.0 - alt_p
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["motion"], prev_az["motion"], alt_p, az_p) if prev_alt["motion"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["motion"]["photons"].append(collected)
                metrics["motion"]["sigma"].append(sig)
                metrics["motion"]["zenith_angles"].append(za)
                metrics["motion"]["extinctions"].append(ext)
                metrics["motion"]["slew_times"].append(slew)
                metrics["motion"]["expose_times"].append(expose_t)
                metrics["motion"]["frame_indices"].append(i)
                metrics["motion"]["alt_az"].append((alt_p, az_p))
                metrics["motion"]["xy_km"].append((xp, yp))
                prev_alt["motion"], prev_az["motion"] = alt_p, az_p
        # scheduler
        obs = matched_sched[i]
        xs, ys, alt_s, az_s = radec_to_altaz_xy(obs['fieldRA'], obs['fieldDec'], all_metas[i]["time"])
        if xs is not None:
            ext = get_value_at_position(grid, xs, ys)
            sig = get_value_at_position(sgrid, xs, ys)
            za = 90.0 - alt_s
            if not np.isnan(ext):
                slew = calculate_slew_time(prev_alt["scheduler"], prev_az["scheduler"], alt_s, az_s) if prev_alt["scheduler"] is not None else 0.0
                expose_t = max(0.0, SLOT_TIME - slew)
                collected = compute_photon_collection(ext) * (expose_t / EXPOSURE_TIME)
                metrics["scheduler"]["photons"].append(collected)
                metrics["scheduler"]["sigma"].append(sig)
                metrics["scheduler"]["zenith_angles"].append(za)
                metrics["scheduler"]["extinctions"].append(ext)
                metrics["scheduler"]["slew_times"].append(slew)
                metrics["scheduler"]["expose_times"].append(expose_t)
                metrics["scheduler"]["frame_indices"].append(i)
                metrics["scheduler"]["alt_az"].append((alt_s, az_s))
                metrics["scheduler"]["xy_km"].append((xs, ys))
                prev_alt["scheduler"], prev_az["scheduler"] = alt_s, az_s
    return metrics

def create_metrics_plot(metrics, output_pdf, output_png):
    """Plot only raw metrics: 3 rows of subplots (no bottom row, no corrected data)."""
    strategies = ("absolute", "motion", "scheduler")
    colors = ("green", "blue", "red")
    labels = ("Absolute Min", "Motion Tracking", "Scheduler")

    fig = plt.figure(figsize=(18, 10))   # reduced height because bottom row removed
    gs = fig.add_gridspec(3, 3, hspace=0.40, wspace=0.32)

    # Row 0: Zenith angle, slew time, sigma
    ax = fig.add_subplot(gs[0,0])
    for s,c,l in zip(strategies, colors, labels):
        if metrics[s]["zenith_angles"]:
            ax.hist(metrics[s]["zenith_angles"], bins=20, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Zenith Angle (°)"); ax.set_ylabel("Count")
    ax.set_title("Zenith Angle Distribution", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0,1])
    for s,c,l in zip(strategies, colors, labels):
        slew = metrics[s]["slew_times"][1:]
        if slew:
            ax.hist(slew, bins=30, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.set_xlabel("Slew Time (s)"); ax.set_ylabel("Count")
    ax.set_title("Slew Time Distribution", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0,2])
    for s,c,l in zip(strategies, colors, labels):
        sigs = [x for x in metrics[s]["sigma"] if not np.isnan(x)]
        if sigs:
            ax.hist(sigs, bins=30, alpha=0.6, color=c, label=l, edgecolor='black', linewidth=0.5)
    ax.axvline(MAX_SIGMA_MAG, color='k', linestyle='--', lw=1.5, label=f"Mask threshold ({MAX_SIGMA_MAG})")
    ax.set_xlabel("Pixel σ uncertainty (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Uncertainty", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)

    # Row 1: Photon timeline (raw only)
    ax = fig.add_subplot(gs[1,:])
    for s,c,l in zip(strategies, colors, labels):
        fi = metrics[s]["frame_indices"]
        ph = metrics[s]["photons"]
        if fi:
            ax.plot(fi, ph, color=c, alpha=0.7, linewidth=1.5, label=l)
    ax.set_xlabel("Frame Number (Time →)"); ax.set_ylabel("Slew-gated photons per slot")
    ax.set_title("Photon Collection Over the Night", weight="bold")
    ax.legend(fontsize=12); ax.grid(alpha=0.3)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

    # Row 2: Total photons, median depth, efficiency (raw only, no corrected bars)
    ax = fig.add_subplot(gs[2,0])
    x_pos = np.arange(len(strategies))
    width = 0.35
    totals_raw = [np.sum(metrics[s]["photons"]) if metrics[s]["photons"] else 0 for s in strategies]
    ax.bar(x_pos, totals_raw, width, color=colors, alpha=0.7, edgecolor='black', linewidth=1.2, label='raw')
    ax.set_xticks(x_pos); ax.set_xticklabels(labels)
    ax.set_ylabel("Total Photons"); ax.set_title("Total Night Photon Collection", weight="bold")
    ax.legend(fontsize=10); ax.grid(alpha=0.3, axis='y')
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))

    ax = fig.add_subplot(gs[2,1])
    depths = []
    for s in strategies:
        mags = [photons_to_magnitude(p) for p in metrics[s]["photons"]]
        mags = [m for m in mags if not np.isnan(m)]
        depths.append(np.median(mags) if mags else 0)
    bars = ax.bar(labels, depths, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for bar, val in zip(bars, depths):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.2f}", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_ylabel("Median 5σ Depth (mag)"); ax.set_title("Survey Depth (slew-scaled)", weight="bold")
    ax.grid(alpha=0.3, axis='y'); ax.invert_yaxis()

    ax = fig.add_subplot(gs[2,2])
    effs = []
    for s in strategies:
        et = np.array(metrics[s]["expose_times"])
        slots = len(et) * SLOT_TIME
        effs.append(np.sum(et) / slots * 100 if slots>0 else 0)
    bars = ax.bar(labels, effs, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    for bar, val in zip(bars, effs):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.1f}%", ha='center', va='bottom', fontsize=12, weight='bold')
    ax.set_ylabel("Shutter-open efficiency (%)"); ax.set_title("Observation Efficiency", weight="bold")
    ax.grid(alpha=0.3, axis='y'); ax.set_ylim(0,105)

    plt.suptitle("Rubin Observatory Pointing Strategy Comparison\n(σ-masked, slew-gated, moon-avoided, zenith-weighted)",
                 fontsize=16, weight="bold", y=0.98)
    plt.savefig(output_pdf, dpi=150, bbox_inches="tight")
    plt.savefig(output_png, dpi=150, bbox_inches="tight")
    print(f"Metrics plot saved: {output_pdf} and {output_png}")
    plt.close()

def save_metrics_data(metrics, output_csv):
    rows = []
    for strategy in ["absolute","motion","scheduler"]:
        n = len(metrics[strategy]["photons"])
        for i in range(n):
            row = {
                "strategy": strategy,
                "frame_idx": metrics[strategy]["frame_indices"][i],
                "photons_raw": metrics[strategy]["photons"][i],
                "extinction": metrics[strategy]["extinctions"][i],
                "sigma": metrics[strategy]["sigma"][i],
                "zenith_angle": metrics[strategy]["zenith_angles"][i],
                "slew_time": metrics[strategy]["slew_times"][i],
                "expose_time": metrics[strategy]["expose_times"][i],
            }
            rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"Metrics data saved to {output_csv}")

def process_night(csv_file, db_file, target_date, start_frame=210):
    """Process a single night and produce plots and data."""
    print(f"\n{'='*60}\nProcessing night: {target_date}\n{'='*60}")

    # Load frames
    try:
        df_frames = load_night_sys_frames(csv_file, target_date, start_frame)
    except ValueError as e:
        print(f"Skipping {target_date}: {e}")
        return
    if len(df_frames) < 5:
        print(f"Skipping {target_date}: only {len(df_frames)} frames")
        return

    # Load scheduler observations
    scheduler_obs = load_scheduler_observations(db_file)

    # Load all grids, sigma, metas
    print("Loading all frames...")
    all_grids = []
    all_sigma = []
    all_metas = []
    for _, row in df_frames.iterrows():
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            obstime = row[TIME_COL].to_pydatetime()
            grid, _, _, _, _ = healpix_to_grid(clouds, obstime)
            sig_grid, _, _, _, _ = healpix_to_grid(sigma, obstime)
            all_grids.append(grid)
            all_sigma.append(sig_grid)
            all_metas.append({"time": obstime})
        except Exception as e:
            print(f"Failed to load frame: {e}")
    print(f"Loaded {len(all_grids)} valid frames.")

    if len(all_grids) < 2:
        print("Not enough grids, skipping.")
        return

    # Match scheduler
    matched_sched = match_scheduler_to_frames(scheduler_obs, [m["time"] for m in all_metas])

    # Motion tracking
    print("Running cloud motion tracking...")
    abs_pos, pred_pos, motion_v, moon_pos = compute_all_positions(all_grids, all_metas)

    # Compute metrics
    print("Computing metrics...")
    metrics = compute_metrics(all_grids, all_sigma, all_metas, abs_pos, pred_pos, moon_pos, matched_sched)

    # Create plots
    create_metrics_plot(metrics,
                        output_pdf=f"metrics_{target_date}.pdf",
                        output_png=f"metrics_{target_date}.png")

    # Save data
    save_metrics_data(metrics, output_csv=f"metrics_data_{target_date}.csv")

    # Summary
    print("\n=== Summary ===")
    for s in ["absolute","motion","scheduler"]:
        ph = np.array(metrics[s]["photons"])
        if len(ph):
            med_depth = np.median([photons_to_magnitude(p) for p in ph if not np.isnan(p)])
            print(f"{s:10s}: {len(ph)} slots, total photons = {np.sum(ph):.3e}, median depth = {med_depth:.2f} mag")
        else:
            print(f"{s:10s}: no valid slots")

def main():
    csv_file = "feb5_data.csv"
    db_file = "baseline_v5.1.0_10yrs.db"
    start_frame = 210

    nights = [
        "2025-09-12",
        "2025-08-12",
        "2025-08-09",
        "2025-07-30",
        "2025-07-29",
        "2025-07-23",
        "2025-07-21",
        "2025-07-20",
        "2025-07-15",
        "2025-07-14",
        "2025-07-01",
        "2025-06-30",
        "2025-06-27"
    ]

    for night in nights:
        process_night(csv_file, db_file, night, start_frame)

if __name__ == "__main__":
    main()


Processing night: 2025-09-12
Loaded 1002 frames for 2025-09-12 (starting at original frame 210)
Scheduler night 572 with 1139 obs
Loading all frames...
Loaded 1002 valid frames.
Running cloud motion tracking...
Computing metrics...
Metrics plot saved: metrics_2025-09-12.pdf and metrics_2025-09-12.png
Metrics data saved to metrics_data_2025-09-12.csv

=== Summary ===
absolute  : 1002 slots, total photons = 5.828e+07, median depth = 15.73 mag
motion    : 1002 slots, total photons = 6.715e+07, median depth = 15.70 mag
scheduler : 77 slots, total photons = 4.395e+06, median depth = 15.81 mag

Processing night: 2025-08-12
Loaded 1094 frames for 2025-08-12 (starting at original frame 210)
Scheduler night 572 with 1139 obs
Loading all frames...
Loaded 1094 valid frames.
Running cloud motion tracking...
Computing metrics...
Metrics plot saved: metrics_2025-08-12.pdf and metrics_2025-08-12.png
Metrics data saved to metrics_data_2025-08-12.csv

=== Summary ===
absolute  : 1094 slots, total phot